## Install Dependencies

In [ ]:
!pip install -q numpy matplotlib scikit-image torch scipy astra-toolbox pydicom

## Core Imports & Global Configuration

In [ ]:
import time
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import astra
import pydicom
from pydicom.data import get_testdata_file
from skimage.data import shepp_logan_phantom
from skimage.transform import resize
from skimage.draw import rectangle
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import random

# Function to set global random seed
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Global Configurations
VOL_SIZE       = 256
DETECTOR_COUNT = 384
ANGLES_FULL    = 180

full_angles_ref = np.linspace(0, np.pi, ANGLES_FULL, endpoint=False)
vol_geom        = astra.create_vol_geom(VOL_SIZE, VOL_SIZE)

HAS_GPU = torch.cuda.is_available()
DEVICE  = torch.device('cuda' if HAS_GPU else 'cpu')
print(f"CUDA Available : {HAS_GPU}")
print(f"Device         : {DEVICE}")
if HAS_GPU:
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

# Set a global seed for reproducibility
set_seed(42)

## Shared Utility Functions

In [ ]:
# Metrics
def calculate_metrics(gt, recon):
    """Compute PSNR and SSIM; normalises reconstruction to [0,1] first."""
    recon_norm = (recon - recon.min()) / (recon.max() - recon.min() + 1e-8)
    val_psnr = psnr(gt, recon_norm, data_range=1.0)
    val_ssim = ssim(gt, recon_norm, data_range=1.0)
    return val_psnr, val_ssim


# Sinogram generation via ASTRA
def generate_sinogram(image, angles):
    """
    Forward-project `image` at the given angles using ASTRA parallel-beam.
    Returns sinogram as a numpy array of shape (len(angles), DETECTOR_COUNT).
    """
    proj_geom = astra.create_proj_geom('parallel', 1.0, DETECTOR_COUNT, angles)
    proj_id   = astra.create_projector('linear', proj_geom, vol_geom)
    sino_id, sinogram = astra.create_sino(image, proj_id)
    sinogram = sinogram.copy()
    astra.data2d.delete(sino_id)
    astra.projector.delete(proj_id)
    return sinogram


# Angle set definitions
def get_angle_sets():
    """Return the four acquisition geometry angle arrays."""
    np.random.seed(42)
    angles_full    = np.linspace(0, np.pi, 180, endpoint=False)
    angles_uniform = np.linspace(0, np.pi, 20,  endpoint=False)
    angles_random  = np.sort(np.random.choice(full_angles_ref, 20, replace=False))
    angles_wedge   = np.concatenate((
        np.linspace(0,          np.pi/3, 60, endpoint=False),
        np.linspace(2*np.pi/3, np.pi,   60, endpoint=False)
    ))
    return {
        'Full View'          : angles_full,
        'Uniform Sparse View': angles_uniform,
        'Random Sparse View' : angles_random,
        'Missing Wedge'      : angles_wedge,
    }

print("Shared utilities loaded.")

## Dataset Loaders

In [ ]:
def load_shepp_logan():
    """Load and normalise the Shepp-Logan phantom to VOL_SIZE × VOL_SIZE."""
    img = shepp_logan_phantom()
    img = resize(img, (VOL_SIZE, VOL_SIZE), anti_aliasing=True)
    img = (img - img.min()) / (img.max() - img.min())
    return img.astype(np.float32)


def create_checkerboard(size=VOL_SIZE, num_squares=8):
    """Create a binary checkerboard image."""
    img  = np.zeros((size, size), dtype=np.float32)
    step = size // num_squares
    for y in range(0, size, step):
        for x in range(0, size, step):
            if (x // step + y // step) % 2 == 1:
                rr, cc = rectangle((y, x), extent=(step, step), shape=img.shape)
                img[rr, cc] = 1.0
    return img


def load_dicom():
    """Load the built-in pydicom CT test file, resize, and normalise."""
    filename = get_testdata_file("CT_small.dcm")
    dataset  = pydicom.dcmread(filename)
    img      = dataset.pixel_array.astype(np.float64)
    img      = resize(img, (VOL_SIZE, VOL_SIZE), anti_aliasing=True)
    img      = (img - img.min()) / (img.max() - img.min())
    return img.astype(np.float32)


# Pre-load all three datasets
datasets = {
    'Shepp-Logan'  : load_shepp_logan(),
    'Checkerboard' : create_checkerboard(),
    'DICOM'        : load_dicom(),
}
print("Datasets loaded:")
for name, img in datasets.items():
    print(f"  {name:15s}  shape={img.shape}  min={img.min():.3f}  max={img.max():.3f}")

---
## 2D Gaussian Representation

### `Gaussians2D` — Core Representation Class

Each Gaussian Gᵢ carries learnable parameters stored as `torch.nn.Parameter`:
| Parameter | Shape | Meaning |
|-----------|-------|---------|
| `mu`      | (N,2) | Centre (x, y) in [0,1] normalised space |
| `log_s`   | (N,2) | Log-scale (→ exp gives sx, sy > 0) |
| `theta`   | (N,1) | Rotation angle in radians |
| `t_raw`   | (N,1) | Pre-sigmoid intensity |

Covariance decomposition:

$Σ = R S Sᵀ Rᵀ$

$R = [cosθ, −sinθ],[sinθ, cosθ]$

$S = diag(sx, sy)$

In [ ]:
class Gaussians2D(torch.nn.Module):
    """
    Learnable 2D Gaussian field for CT reconstruction.

    Parameters (all torch.nn.Parameter):
        mu     (N, 2)  — centres in normalised [0,1] space
        log_s  (N, 2)  — log-scale → exp gives sx, sy > 0
        theta  (N, 1)  — rotation angle in radians
        t_raw  (N, 1)  — pre-sigmoid intensity
    """

    def __init__(self, mu_init, s_init, t_init, device=DEVICE):
        super().__init__()
        N = mu_init.shape[0]
        self.mu    = torch.nn.Parameter(torch.tensor(mu_init,        dtype=torch.float32, device=device))
        self.log_s = torch.nn.Parameter(torch.tensor(np.log(s_init), dtype=torch.float32, device=device))

        theta_init = (torch.rand(N, 1, device=device) - 0.5) * (np.pi / 3)
        self.theta = torch.nn.Parameter(theta_init)

        self.t_raw = torch.nn.Parameter(torch.tensor(
            np.log(np.clip(t_init, 1e-6, 1-1e-6) / (1.0 - np.clip(t_init, 1e-6, 1-1e-6))).reshape(-1, 1),
            dtype=torch.float32, device=device))

        # Gradient accumulation buffer
        self.grad_accum       = torch.zeros(N, device=device)
        self.grad_accum_steps = 0

        # Index cache
        self._flat_idx    = None
        self._gauss_ids   = None
        self._cache_grid  = None
        self._index_dirty = True

    def accumulate_grad(self):
        """
        Called after every loss.backward() + clip, before optimizer.step().
        Accumulates |∇μ| for each Gaussian over the ADC window.
        """
        if self.mu.grad is not None:
            with torch.no_grad():
                self.grad_accum += self.mu.grad.norm(dim=1)
            self.grad_accum_steps += 1

    def reset_grad_accum(self):
        """Zero the accumulation buffer. Called after each ADC pass."""
        self.grad_accum.zero_()\


        self.grad_accum_steps = 0

    @property
    def scales(self):
        return torch.exp(self.log_s)

    @property
    def intensities(self):
        return torch.sigmoid(self.t_raw)

    def get_cov_inv(self):
        """Σ⁻¹ = R diag(1/sx², 1/sy²) Rᵀ, shape (N, 2, 2)."""
        s   = self.scales
        cos = torch.cos(self.theta)
        sin = torch.sin(self.theta)

        si0_sq = 1.0 / (s[:, 0:1] ** 2 + 1e-8)
        si1_sq = 1.0 / (s[:, 1:2] ** 2 + 1e-8)

        c00 = cos**2 * si0_sq + sin**2 * si1_sq
        c01 = cos * sin * (si0_sq - si1_sq)
        c11 = sin**2 * si0_sq + cos**2 * si1_sq

        return torch.stack([
            torch.cat([c00, c01], dim=1),
            torch.cat([c01, c11], dim=1),
        ], dim=1)

    def _rebuild_index_cache(self, grid_size):
        device = self.mu.device
        G      = grid_size
        N      = self.mu.shape[0]

        with torch.no_grad():
            mu_px  = (self.mu * (G - 1)).round().long().clamp(0, G - 1)
            cx_np  = mu_px[:, 0].cpu().numpy().astype(np.int32)
            cy_np  = mu_px[:, 1].cpu().numpy().astype(np.int32)
            sig_px = self.scales.max(dim=1).values * (G - 1)
            r_np   = sig_px.mul(3.0).ceil().long().clamp(min=1).cpu().numpy().astype(np.int32)

        flat_list = []
        gids_list = []

        for r in np.unique(r_np):
            group  = np.where(r_np == r)[0].astype(np.int32)
            off    = np.arange(-r, r + 1, dtype=np.int32)
            ro, co = np.meshgrid(off, off, indexing='ij')
            ro = ro.ravel(); co = co.ravel()

            rows = cy_np[group, None] + ro[None, :]
            cols = cx_np[group, None] + co[None, :]

            valid = (rows >= 0) & (rows < G) & (cols >= 0) & (cols < G)
            rows  = np.clip(rows, 0, G - 1)
            cols  = np.clip(cols, 0, G - 1)
            flat  = rows * G + cols
            gids  = np.broadcast_to(group[:, None], flat.shape)

            flat_list.append(flat[valid])
            gids_list.append(gids[valid])

        flat_np = np.concatenate(flat_list).astype(np.int64)
        gids_np = np.concatenate(gids_list).astype(np.int64)

        self._flat_idx    = torch.from_numpy(flat_np).to(device)
        self._gauss_ids   = torch.from_numpy(gids_np).to(device)
        self._cache_grid  = grid_size
        self._index_dirty = False

    def voxelise(self, grid_size=VOL_SIZE):
        device = self.mu.device
        G      = grid_size

        if self._index_dirty or self._cache_grid != G:
            self._rebuild_index_cache(G)

        flat_idx  = self._flat_idx
        gauss_ids = self._gauss_ids

        row_idx = flat_idx // G
        col_idx = flat_idx %  G

        px_x = col_idx.float() / (G - 1)
        px_y = row_idx.float() / (G - 1)

        mu_m  = self.mu[gauss_ids]
        ci_m  = self.get_cov_inv()[gauss_ids]
        t_m   = self.intensities.squeeze(1)[gauss_ids]

        dx   = px_x - mu_m[:, 0]
        dy   = px_y - mu_m[:, 1]
        diff = torch.stack([dx, dy], dim=1).unsqueeze(2)

        D2   = torch.bmm(torch.bmm(diff.transpose(1, 2), ci_m), diff).squeeze(-1).squeeze(-1)
        vals = t_m * torch.exp(-0.5 * D2)

        volume = torch.zeros(G * G, dtype=torch.float32, device=device)
        volume.scatter_add_(0, flat_idx, vals)

        return volume.view(G, G)


print("Gaussians2D defined.")

### FBP-Guided Gaussian Initialisation

In [ ]:
def fbp_init_gaussians(fbp_image, n_gaussians=8000, tau=0.05,
                       k_sigma=0.25, k_t=0.5, neighbor_radius=5,
                       interior_fraction=0.20):
    """
    FBP-image-guided Gaussian initialisation (Section 3.1 of 3DGR-CT).

    Args:
        fbp_image         : 2D numpy array, normalised [0,1]
        n_gaussians       : target number of Gaussians
        tau               : background threshold
        k_sigma           : scale coefficient for covariance init
        k_t               : intensity multiplier on FBP values
        neighbor_radius   : radius (pixels) for neighbour density estimate
        interior_fraction : fraction of Gaussians placed uniformly in foreground
                            regardless of FBP intensity (default 0.20)

    Returns:
        mu_init : (N, 2) float32 — centres in [0,1]
        s_init  : (N, 2) float32 — isotropic scales
        t_init  : (N,)   float32 — intensities
    """
    H, W = fbp_image.shape

    # Step 1: foreground mask
    foreground = fbp_image > tau

    # Step 2: gradient magnitude (for sigma init only)
    gy, gx   = np.gradient(fbp_image)
    grad_mag = np.sqrt(gx**2 + gy**2)
    grad_mag[~foreground] = 0.0

    # Step 3: split budget into FBP-weighted and uniform-interior
    n_interior = int(n_gaussians * interior_fraction)
    n_fbp      = n_gaussians - n_interior

    candidate_ys, candidate_xs = np.where(foreground)
    if len(candidate_ys) == 0:
        candidate_ys = np.random.randint(0, H, n_gaussians)
        candidate_xs = np.random.randint(0, W, n_gaussians)
        n_interior   = 0
        n_fbp        = n_gaussians

    # FBP-weighted sample (same as before, from foreground pixels)
    n_cands = len(candidate_ys)
    idx_fbp = np.random.choice(n_cands, n_fbp, replace=(n_cands < n_fbp))
    ys_fbp  = candidate_ys[idx_fbp]
    xs_fbp  = candidate_xs[idx_fbp]

    # Uniform-interior sample: full foreground mask, ignoring intensity
    # This is the same mask but sampled uniformly — ensures interior coverage.
    if n_interior > 0:
        idx_int = np.random.choice(n_cands, n_interior, replace=(n_cands < n_interior))
        ys_int  = candidate_ys[idx_int]
        xs_int  = candidate_xs[idx_int]
        ys = np.concatenate([ys_fbp, ys_int])
        xs = np.concatenate([xs_fbp, xs_int])
    else:
        ys, xs = ys_fbp, xs_fbp

    N = len(ys)

    # Step 3b: Gaussian jitter
    jitter_sigma = 1.5 / max(H, W)
    jitter_x = np.random.normal(0.0, jitter_sigma, N)
    jitter_y = np.random.normal(0.0, jitter_sigma, N)

    # Step 4: normalise centres to [0,1]
    mu_x    = np.clip(xs / (W - 1) + jitter_x, 0.0, 1.0)
    mu_y    = np.clip(ys / (H - 1) + jitter_y, 0.0, 1.0)
    mu_init = np.stack([mu_x, mu_y], axis=1).astype(np.float32)

    # Step 5: neighbour-density covariance init
    from scipy.spatial import cKDTree
    tree        = cKDTree(mu_init)
    n_neighbors = np.array(
        [len(tree.query_ball_point(mu_init[i], r=neighbor_radius / (W - 1)))
         for i in range(N)], dtype=np.float32)
    n_neighbors = np.maximum(n_neighbors, 1.0)
    sigma_vals  = k_sigma / n_neighbors
    sigma_vals  = np.clip(sigma_vals, 5e-4, 0.015)
    s_init      = np.stack([sigma_vals, sigma_vals], axis=1).astype(np.float32)

    # Step 6: intensity init
    # FBP portion: use FBP pixel value (as before)
    # Interior uniform portion: use small constant (0.1) — FBP interior is
    # near-zero which would initialise t_init≈0 and make these Gaussians
    # invisible at iter 0. A small constant gives them gradient signal.
    t_fbp  = np.clip(k_t * fbp_image[ys_fbp, xs_fbp].astype(np.float32), 1e-4, 0.9999)
    if n_interior > 0:
        t_int  = np.full(n_interior, 0.3, dtype=np.float32)
        t_init = np.concatenate([t_fbp, t_int])
    else:
        t_init = t_fbp

    return mu_init, s_init, t_init


print("FBP-guided initialisation defined.")

### Differentiable CT Projector
Wraps ASTRA's forward projection inside a `torch.autograd.Function` so gradients flow back through the sinogram into the Gaussian parameters. This is the 2D equivalent of the differentiable CT projector in Section 3.2 of 3DGR-CT.

In [ ]:
class DifferentiableCTProjector(torch.autograd.Function):
    """
    Differentiable parallel-beam CT projector backed by ASTRA.

    Forward  : volume (H×W tensor) → sinogram (A×D tensor)
    Backward : grad_sinogram        → grad_volume via BP_CUDA / BP
    """

    @staticmethod
    def forward(ctx, volume_tensor, angles):
        ctx.angles = angles
        ctx.vol_size = volume_tensor.shape[-1] # Store volume size
        device     = volume_tensor.device

        vol_np  = volume_tensor.detach().cpu().numpy().astype(np.float32)
        sino_np = _astra_fp(vol_np, angles)

        return torch.tensor(sino_np, dtype=torch.float32, device=device)

    @staticmethod
    def backward(ctx, grad_sinogram):
        angles = ctx.angles
        vol_size = ctx.vol_size # Retrieve volume size
        device = grad_sinogram.device

        grad_sino_np = grad_sinogram.detach().cpu().numpy().astype(np.float32)
        grad_vol_np  = _astra_bp(grad_sino_np, angles, vol_size) # Pass vol_size

        return torch.tensor(grad_vol_np, dtype=torch.float32, device=device), None


def _astra_fp(vol_np, angles):
    """ASTRA forward projection (FP_CUDA when available)."""
    H, W      = vol_np.shape
    proj_geom = astra.create_proj_geom('parallel', 1.0, DETECTOR_COUNT, angles)
    vol_geom_ = astra.create_vol_geom(H, W)

    # Use CUDA projector when available — silent CPU fallback was the bug
    if HAS_GPU:
        vol_id  = astra.data2d.create('-vol',  vol_geom_,  vol_np)
        sino_id = astra.data2d.create('-sino', proj_geom,  0)
        cfg     = astra.astra_dict('FP_CUDA')
        cfg['VolumeDataId']     = vol_id
        cfg['ProjectionDataId'] = sino_id
        alg_id  = astra.algorithm.create(cfg)
        astra.algorithm.run(alg_id)
        sino = astra.data2d.get(sino_id).copy()
        astra.algorithm.delete(alg_id)
        astra.data2d.delete(sino_id)
        astra.data2d.delete(vol_id)
    else:
        proj_id = astra.create_projector('linear', proj_geom, vol_geom_)
        sino_id, sino = astra.create_sino(vol_np, proj_id)
        sino = sino.copy()
        astra.data2d.delete(sino_id)
        astra.projector.delete(proj_id)

    return sino.astype(np.float32)


def _astra_bp(grad_sino_np, angles, vol_size): # Accept vol_size
    """ASTRA back-projection adjoint (BP_CUDA when available)."""
    proj_geom = astra.create_proj_geom('parallel', 1.0, DETECTOR_COUNT, angles)
    vol_geom_ = astra.create_vol_geom(vol_size, vol_size) # Use vol_size here

    if HAS_GPU:
        sino_id = astra.data2d.create('-sino', proj_geom,  grad_sino_np)
        rec_id  = astra.data2d.create('-vol',  vol_geom_,  0.0)
        cfg     = astra.astra_dict('BP_CUDA')
        cfg['ReconstructionDataId'] = rec_id
        cfg['ProjectionDataId']     = sino_id
        alg_id  = astra.algorithm.create(cfg)
        astra.algorithm.run(alg_id)
        grad_vol = astra.data2d.get(rec_id).copy()
        astra.algorithm.delete(alg_id)
        astra.data2d.delete(rec_id)
        astra.data2d.delete(sino_id)
    else:
        sino_id  = astra.data2d.create('-sino', proj_geom,  grad_sino_np)
        rec_id   = astra.data2d.create('-vol',  vol_geom_,  0.0)
        proj_id  = astra.create_projector('linear', proj_geom, vol_geom_)
        cfg      = astra.astra_dict('BP')
        cfg['ReconstructionDataId'] = rec_id
        cfg['ProjectionDataId']     = sino_id
        cfg['ProjectorId']          = proj_id
        alg_id   = astra.algorithm.create(cfg)
        astra.algorithm.run(alg_id)
        grad_vol = astra.data2d.get(rec_id).copy()
        astra.algorithm.delete(alg_id)
        astra.data2d.delete(rec_id)
        astra.data2d.delete(sino_id)
        astra.projector.delete(proj_id)

    return grad_vol.astype(np.float32)


def project(volume_tensor, angles):
    """
    Differentiable entry point.
    volume_tensor : (H, W) float32 torch tensor
    angles        : numpy array of projection angles
    Returns       : (A, D) sinogram tensor, same device
    """
    return DifferentiableCTProjector.apply(volume_tensor, angles)


print("Differentiable CT projector (FP_CUDA / BP_CUDA) defined.")

### Adaptive Density Control
Clone / Split / Prune operations from Section 3.3 of 3DGR-CT, adapted to 2D.
| Operation | Trigger | Action |
|-----------|---------|--------|
| **Clone** | High $‖∇μ‖$, small scale | Duplicate; one moves along gradient |
| **Split** | High $‖∇μ‖$, large scale | Replace with 2 smaller Gaussians (x0.8 scale, ÷2 intensity) |
| **Prune** | Near-zero intensity | Remove |

In [ ]:
def adaptive_density_control(gaussians, densify_percentile=95,
                              prune_threshold=1e-6,
                              sigma_split_thresh=0.01,
                              max_gaussians=12000):
    """
    Clone / split / prune Gaussians.
    """
    device = gaussians.mu.device

    with torch.no_grad():
        mu    = gaussians.mu.data
        log_s = gaussians.log_s.data
        theta = gaussians.theta.data
        t_raw = gaussians.t_raw.data
        s     = gaussians.scales
        t     = gaussians.intensities
        N     = mu.shape[0]

        if gaussians.grad_accum_steps > 0:
            grad_norm  = gaussians.grad_accum / gaussians.grad_accum_steps
            pct_thresh = torch.quantile(grad_norm, densify_percentile / 100.0)
            high_grad  = grad_norm > pct_thresh
        elif gaussians.mu.grad is not None:
            # Fallback to single-step if buffer is empty (first ADC call)
            grad_norm  = gaussians.mu.grad.norm(dim=1)
            pct_thresh = torch.quantile(grad_norm, densify_percentile / 100.0)
            high_grad  = grad_norm > pct_thresh
        else:
            high_grad = torch.zeros(N, dtype=torch.bool, device=device)

        mean_scale = s.mean(dim=1)
        large_s    = mean_scale > sigma_split_thresh      # fixed at 0.01
        low_t      = t.squeeze(1) < prune_threshold       # 1e-6

        clone_mask = high_grad & ~large_s & ~low_t
        split_mask = high_grad &  large_s & ~low_t
        keep_mask  = ~low_t

        keep_idx  = torch.where(keep_mask)[0]
        clone_idx = torch.where(clone_mask)[0]
        split_idx = torch.where(split_mask)[0]

        n_clone = clone_idx.shape[0]
        n_split = split_idx.shape[0]
        n_prune = low_t.sum().item()

        mu_new    = [mu[keep_mask]]
        log_s_new = [log_s[keep_mask]]
        theta_new = [theta[keep_mask]]
        t_raw_new = [t_raw[keep_mask]]
        source_parts = [keep_idx]

        if clone_mask.any():
            mu_c     = mu[clone_mask]
            # Use accumulated grad direction for clone displacement
            if gaussians.grad_accum_steps > 0:
                grad_c = gaussians.grad_accum[clone_mask].unsqueeze(1) * \
                         gaussians.mu.grad[clone_mask] / \
                         (gaussians.mu.grad[clone_mask].norm(dim=1, keepdim=True) + 1e-8)
            else:
                grad_c = gaussians.mu.grad[clone_mask]
            grad_dir = grad_c / (grad_c.norm(dim=1, keepdim=True) + 1e-8)
            mu_moved = torch.clamp(mu_c + 0.01 * grad_dir, 0.0, 1.0)
            mu_new    += [mu_moved]
            log_s_new += [log_s[clone_mask]]
            theta_new += [theta[clone_mask]]
            t_raw_new += [t_raw[clone_mask]]
            source_parts.append(clone_idx)

        if split_mask.any():
            mu_sp     = mu[split_mask]
            s_sp      = s[split_mask]
            new_log_s = log_s[split_mask] + np.log(0.8)
            t_half    = torch.clamp(
                torch.sigmoid(t_raw[split_mask]) / 2.0, 1e-6, 1 - 1e-6)
            new_t_raw = torch.log(t_half / (1.0 - t_half))
            noise     = torch.randn_like(mu_sp) * s_sp * 0.3
            mu_new    += [torch.clamp(mu_sp + noise, 0.0, 1.0),
                          torch.clamp(mu_sp - noise, 0.0, 1.0)]
            log_s_new += [new_log_s, new_log_s]
            theta_new += [theta[split_mask]] * 2
            t_raw_new += [new_t_raw, new_t_raw]
            source_parts += [split_idx, split_idx]

        mu_cat     = torch.cat(mu_new,    dim=0)
        log_s_cat  = torch.cat(log_s_new, dim=0)
        theta_cat  = torch.cat(theta_new, dim=0)
        t_raw_cat  = torch.cat(t_raw_new, dim=0)
        source_idx = torch.cat(source_parts, dim=0)

        if mu_cat.shape[0] > max_gaussians:
            perm       = torch.randperm(mu_cat.shape[0], device=device)[:max_gaussians]
            mu_cat     = mu_cat[perm]
            log_s_cat  = log_s_cat[perm]
            theta_cat  = theta_cat[perm]
            t_raw_cat  = t_raw_cat[perm]
            source_idx = source_idx[perm]

    new_g = Gaussians2D.__new__(Gaussians2D)
    torch.nn.Module.__init__(new_g)
    new_g.mu    = torch.nn.Parameter(mu_cat.to(device))
    new_g.log_s = torch.nn.Parameter(log_s_cat.to(device))
    new_g.theta = torch.nn.Parameter(theta_cat.to(device))
    new_g.t_raw = torch.nn.Parameter(t_raw_cat.to(device))
    new_g._flat_idx    = None
    new_g._gauss_ids   = None
    new_g._cache_grid  = None
    new_g._index_dirty = True

    new_g.grad_accum       = torch.zeros(mu_cat.shape[0], device=device)
    new_g.grad_accum_steps = 0

    stats = {'clone': n_clone, 'split': n_split, 'prune': n_prune,
             'N_new': mu_cat.shape[0]}
    return new_g, source_idx, stats


print("ADC (accumulated gradient, fixed sigma_split=0.01) defined.")

### Two-Stage Optimiser & Training Loop
- **Stage 1** (low-resolution 128x128): fast coarse placement
- **Stage 2** (full-resolution 256x256): fine-grained refinement

In [ ]:
def make_optimizer(gaussians):
    return torch.optim.Adam([
        {'params': [gaussians.mu],    'lr': 2e-3,  'name': 'mu'},
        {'params': [gaussians.log_s], 'lr': 5e-3,  'name': 'log_s'},
        {'params': [gaussians.theta], 'lr': 1e-3,  'name': 'theta'},
        {'params': [gaussians.t_raw], 'lr': 5e-2,  'name': 't_raw'},
    ], betas=(0.9, 0.999))


def compute_all_lrs(iteration, n_iters):
    def decay(start, end):
        return start * (end / start) ** (iteration / n_iters)
    return {
        'mu':    decay(2e-3, 2e-6),
        'log_s': decay(5e-3, 5e-5),
        'theta': decay(1e-3, 1e-5),
        't_raw': decay(5e-2, 5e-4),
    }


def update_optimizer_state(old_optimizer, new_optimizer, source_idx):
    for old_group, new_group in zip(old_optimizer.param_groups,
                                    new_optimizer.param_groups):
        old_param = old_group['params'][0]
        new_param = new_group['params'][0]
        if old_param not in old_optimizer.state:
            continue
        old_state = old_optimizer.state[old_param]
        new_state = new_optimizer.state[new_param]
        new_state['step'] = old_state['step']
        for key in ('exp_avg', 'exp_avg_sq'):
            if key in old_state:
                new_state[key] = old_state[key][source_idx].clone()
    return new_optimizer


def compute_tv_loss(volume):
    """
    Isotropic Total-Variation regulariser.
    """
    diff_h = volume[1:, :] - volume[:-1, :]
    diff_w = volume[:, 1:] - volume[:, :-1]
    tv = torch.sqrt(diff_h[:, :-1] ** 2 + diff_w[:-1, :] ** 2 + 1e-8)
    return tv.mean()


def compute_scale_reg_loss(gaussians):
    """
    Scale regularisation: penalise excessively small Gaussians.

    Args:
        gaussians : Gaussians2D object (must be called within training loop
                    so parameters are on the correct device)

    Returns:
        scalar scale regularisation loss
    """
    scale_min  = 0.004   # ~1 pixel at 256px resolution
    mean_scale = gaussians.scales.mean(dim=1)   # (N,)
    deficit    = torch.clamp(scale_min - mean_scale, min=0.0)
    return (deficit ** 2).mean()


print("Optimizer, LR schedule, TV loss, scale regularisation defined.")

In [ ]:
def train_gaussian_reconstruction(
    image_np,
    angles,
    n_gaussians        = 8000,
    n_iters            = 10000,
    adc_start          = 500,
    adc_every          = 100,
    adc_stop           = 2500,
    densify_percentile = 95,
    prune_thresh       = 1e-6,
    max_gaussians      = 12000,
    lambda_tv          = 1e-3,
    verbose            = True,
):

    device = DEVICE

    sino_gt_np = generate_sinogram(image_np, angles)
    sino_max   = float(sino_gt_np.max()) if sino_gt_np.max() > 0 else 1.0
    sino_gt    = torch.tensor(sino_gt_np / sino_max, dtype=torch.float32, device=device)

    proj_geom   = astra.create_proj_geom('parallel', 1.0, DETECTOR_COUNT, angles)
    proj_id_fbp = astra.create_projector('linear', proj_geom, vol_geom)
    sino_id_fbp, _ = astra.create_sino(image_np, proj_id_fbp)

    rec_id_fbp = astra.data2d.create('-vol', vol_geom, 0.0)
    cfg_fbp    = astra.astra_dict('FBP_CUDA' if HAS_GPU else 'FBP')
    cfg_fbp['ReconstructionDataId'] = rec_id_fbp
    cfg_fbp['ProjectionDataId']     = sino_id_fbp
    if not HAS_GPU:
        cfg_fbp['ProjectorId'] = proj_id_fbp
    alg_fbp = astra.algorithm.create(cfg_fbp)
    astra.algorithm.run(alg_fbp)
    fbp_image = astra.data2d.get(rec_id_fbp).copy()
    astra.algorithm.delete(alg_fbp)
    astra.data2d.delete(rec_id_fbp)
    astra.data2d.delete(sino_id_fbp)
    astra.projector.delete(proj_id_fbp)

    fbp_image = (fbp_image - fbp_image.min()) / \
                (fbp_image.max() - fbp_image.min() + 1e-8)
    fbp_image = fbp_image.astype(np.float32)

    mu_init, s_init, t_init = fbp_init_gaussians(fbp_image, n_gaussians=n_gaussians)
    gaussians = Gaussians2D(mu_init, s_init, t_init, device=device)
    optimizer = make_optimizer(gaussians)

    loss_curve      = []
    psnr_curve      = []
    loss_sino_curve = []
    loss_tv_curve   = []
    start_time      = time.time()

    adc_consecutive_clone_only = 0
    adc_silenced               = False


    best_psnr  = -np.inf
    best_recon = None

    for iteration in range(n_iters):

        all_lrs = compute_all_lrs(iteration, n_iters)
        for group in optimizer.param_groups:
            group['lr'] = all_lrs[group['name']]

        optimizer.zero_grad()
        volume    = gaussians.voxelise(grid_size=VOL_SIZE)
        sino_pred = project(volume, angles) / sino_max

        loss_sino = torch.mean((sino_pred - sino_gt) ** 2)

        if iteration > adc_stop and lambda_tv > 0:
            loss_tv = lambda_tv * compute_tv_loss(volume)
        else:
            loss_tv = torch.tensor(0.0, device=device)

        loss = loss_sino + loss_tv
        loss.backward()

        all_params = [gaussians.mu, gaussians.log_s,
                      gaussians.theta, gaussians.t_raw]
        torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
        gaussians.accumulate_grad()
        optimizer.step()

        # ADC with post-hoc early exit
        adc_due = (iteration >= adc_start
                   and iteration <= adc_stop
                   and (iteration % adc_every) == 0
                   and not adc_silenced)

        if adc_due:
            if adc_consecutive_clone_only >= 2:
                # All large Gaussians exhausted; further calls are pure churn.
                # Silence ADC for the rest of the densify window.
                adc_silenced = True
                if verbose:
                    print(f"    [ADC iter {iteration}] STOPPED — "
                          f"2 consecutive clone-only calls, churn suppressed")
            else:
                old_optimizer = optimizer
                gaussians, source_idx, stats = adaptive_density_control(
                    gaussians,
                    densify_percentile=densify_percentile,
                    prune_threshold=prune_thresh,
                    sigma_split_thresh=0.01,
                    max_gaussians=max_gaussians,
                )
                optimizer = make_optimizer(gaussians)
                optimizer = update_optimizer_state(old_optimizer, optimizer,
                                                   source_idx)
                for group in optimizer.param_groups:
                    group['lr'] = all_lrs[group['name']]

                # Track consecutive clone-only calls at capacity
                at_cap      = (stats['N_new'] >= max_gaussians)
                clone_only = (stats['split'] < 5 and at_cap)
                # Treat split < 5 as clone-only: a handful of splits on 12,000 Gaussians
                # (< 0.04%) has negligible structural effect and should not reset the
                # counter. In V9, a single split=1 at iter 1200 reset the counter to 0,
                # delaying the early exit by two extra churn calls.
                if clone_only:
                    adc_consecutive_clone_only += 1
                else:
                    adc_consecutive_clone_only = 0

                if verbose:
                    tag = f"  [clone-only #{adc_consecutive_clone_only}]" if clone_only else ""
                    print(f"    [ADC iter {iteration}] "
                        f"clone={stats['clone']}  split={stats['split']}  "
                        f"prune={stats['prune']}  N={stats['N_new']}{tag}")

        loss_curve.append(loss.item())
        loss_sino_curve.append(loss_sino.item())
        loss_tv_curve.append(loss_tv.item())

        with torch.no_grad():
            vol_np = volume.detach().cpu().numpy()
            p, _   = calculate_metrics(image_np, vol_np)
            psnr_curve.append(p)
            if p > best_psnr:
                best_psnr  = p
                best_recon = vol_np.copy()

        if verbose and (iteration % 500 == 0 or iteration == n_iters - 1):
            phase = 'refine' if iteration > adc_stop else 'densify'
            best_marker = " ★" if p >= best_psnr - 0.001 else ""
            print(f"  Iter {iteration:5d}/{n_iters}  [{phase}]  "
                  f"Loss={loss.item():.6f}  "
                  f"(sino={loss_sino.item():.6f}  "
                  f"tv={loss_tv.item():.6f})  "
                  f"PSNR={p:.2f} dB  N={gaussians.mu.shape[0]}{best_marker}")

    with torch.no_grad():
        recon_np = gaussians.voxelise(grid_size=VOL_SIZE).cpu().numpy()

    if best_recon is None:
        best_recon = recon_np

    return (recon_np, best_recon, loss_curve, psnr_curve, time.time() - start_time, gaussians,
            loss_sino_curve, loss_tv_curve)


print("Training loop defined.")

### Visualization Utilities

In [ ]:
ROI_COORDS = {
    'Shepp-Logan'  : (60, 120, 100, 160),   # top-right skull boundary — highest residual region
    'Checkerboard' : (96, 160, 96, 160),     # 4-square corner junction
    'DICOM'        : (60, 124, 170, 234),    # right-side bone/soft tissue boundary
}

print("ROI crop coordinates defined.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.gridspec import GridSpec

def plot_gaussian_experiment(title, image, angles, recon,
                             val_psnr, val_ssim, exec_time,
                             gaussians=None,
                             loss_curve=None, psnr_curve=None,
                             dataset_name=None, line_row_idx=None,
                             loss_sino_curve=None,
                             loss_tv_curve=None,
                             adc_stop_iter=2000):
    """
    Row 1: Ground Truth | Sinogram | Reconstruction | Residual Error Map | Gaussian Centres
    Row 2: ROI Zoom     | PSNR Curve | 1D Profile   | Energy Diff        | Convergence
    """
    G = VOL_SIZE

    sino_pred_np = generate_sinogram(recon, angles)
    padded_sino  = np.zeros((ANGLES_FULL, DETECTOR_COUNT))
    for i, a in enumerate(angles):
        idx = np.abs(full_angles_ref - a).argmin()
        padded_sino[idx, :] = sino_pred_np[i, :]

    recon_norm = (recon - recon.min()) / (recon.max() - recon.min() + 1e-8)
    residual   = np.abs(image - recon_norm)

    roi = ROI_COORDS.get(dataset_name, (80, 160, 80, 160))
    y0, y1, x0, x1 = roi

    row_idx  = line_row_idx if line_row_idx is not None else G // 2
    gt_line  = image[row_idx, :]
    rec_line = recon_norm[row_idx, :]

    gt_fft      = np.fft.fftshift(np.abs(np.fft.fft(gt_line)))
    rec_fft     = np.fft.fftshift(np.abs(np.fft.fft(rec_line)))
    freqs       = np.linspace(-0.5, 0.5, len(gt_fft))
    energy_diff = (rec_fft - gt_fft) / (gt_fft + 1e-8)

    fig = plt.figure(figsize=(25, 10))
    gs  = GridSpec(2, 5, figure=fig, hspace=0.38, wspace=0.32)

    fig.suptitle(
        f"{title}\nPSNR: {val_psnr:.2f} dB  |  SSIM: {val_ssim:.4f}  |  Time: {exec_time:.1f}s",
        fontsize=13, fontweight='bold', y=0.98
    )

    # ── Row 1 ──────────────────────────────────────────────────────────────

    # [0] Ground Truth
    ax0 = fig.add_subplot(gs[0, 0])
    ax0.set_title("Ground Truth", fontweight='bold', fontsize=10)
    ax0.imshow(image, cmap='gray', vmin=0, vmax=1)
    ax0.add_patch(Rectangle((x0, y0), x1-x0, y1-y0,
                             edgecolor='red', facecolor='none', linewidth=1.5))
    ax0.axhline(y=row_idx, color='cyan', linewidth=0.8, linestyle='--', alpha=0.7)
    ax0.axis('off')

    # [1] Sinogram
    ax1 = fig.add_subplot(gs[0, 1])
    ax1.set_title("Sinogram", fontweight='bold', fontsize=10)
    ax1.imshow(padded_sino, cmap='gray', aspect='auto')
    ax1.set_ylabel("Angle index (180°)", fontsize=8)
    ax1.set_xlabel("Detector pixels", fontsize=8)
    ax1.tick_params(labelsize=7)

    # [2] Reconstruction
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.set_title("Reconstruction", fontweight='bold', fontsize=10)
    ax2.imshow(recon_norm, cmap='gray', vmin=0, vmax=1)
    ax2.axis('off')

    # [3] Residual Error Map
    ax3 = fig.add_subplot(gs[0, 3])
    ax3.set_title("Residual Error Map", fontweight='bold', fontsize=10)
    im3 = ax3.imshow(residual, cmap='hot', vmin=0, vmax=residual.max())
    plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
    ax3.axis('off')

    # [4] Gaussian Centres
    ax4 = fig.add_subplot(gs[0, 4])
    ax4.set_title(
        f"Gaussian Centres (N={gaussians.mu.shape[0]:,})" if gaussians is not None
        else "Gaussian Centres",
        fontweight='bold', fontsize=10
    )
    ax4.imshow(recon_norm, cmap='gray', vmin=0, vmax=1, alpha=0.35)
    if gaussians is not None:
        with torch.no_grad():
            mu_np = gaussians.mu.detach().cpu().numpy()
            t_np  = gaussians.intensities.squeeze(1).detach().cpu().numpy()
        sc = ax4.scatter(mu_np[:, 0] * (G - 1), mu_np[:, 1] * (G - 1),
                         c=t_np, cmap='plasma', s=0.4, alpha=0.6, linewidths=0)
        plt.colorbar(sc, ax=ax4, fraction=0.046, pad=0.04, label='Intensity')
    ax4.set_xlim(0, G - 1)
    ax4.set_ylim(G - 1, 0)
    ax4.axis('off')

    # ── Row 2 ──────────────────────────────────────────────────────────────

    # [5] ROI Zoom — GT crop (top) and Recon crop (bottom), stacked
    ax5 = fig.add_subplot(gs[1, 0])
    ax5.set_title(f"ROI Zoom [{y0}:{y1}, {x0}:{x1}]", fontweight='bold', fontsize=10)
    gt_roi  = image[y0:y1, x0:x1]
    rec_roi = recon_norm[y0:y1, x0:x1]
    h_roi   = gt_roi.shape[0]
    # Stack with a 2-pixel white separator
    sep      = np.ones((2, gt_roi.shape[1]), dtype=np.float32)
    stacked  = np.concatenate([gt_roi, sep, rec_roi], axis=0)
    ax5.imshow(stacked, cmap='gray', vmin=0, vmax=1)
    # Label each half
    ax5.text(gt_roi.shape[1] / 2, h_roi / 2,      'GT',
             ha='center', va='center', fontsize=8,
             color='yellow', fontweight='bold')
    ax5.text(gt_roi.shape[1] / 2, h_roi + 2 + h_roi / 2, 'Recon',
             ha='center', va='center', fontsize=8,
             color='yellow', fontweight='bold')
    ax5.axis('off')

    # [6] PSNR Curve
    ax6 = fig.add_subplot(gs[1, 1])
    ax6.set_title("PSNR Curve", fontweight='bold', fontsize=10)
    if psnr_curve is not None:
        iters = np.arange(len(psnr_curve))
        ax6.plot(iters, psnr_curve, color='steelblue', linewidth=1.2)
        ax6.axvline(x=adc_stop_iter, color='gray', linestyle='--', linewidth=0.8, label='ADC stop')
        ax6.set_xlabel("Iteration", fontsize=8)
        ax6.set_ylabel("PSNR (dB)", fontsize=8)
        ax6.tick_params(labelsize=7)
        ax6.legend(fontsize=7)
        ax6.annotate(f'{psnr_curve[-1]:.2f} dB',
                     xy=(len(psnr_curve) - 1, psnr_curve[-1]),
                     xytext=(-48, -12), textcoords='offset points',
                     fontsize=7, color='steelblue')
    else:
        ax6.axis('off')

    # [7] 1D Intensity Profile
    ax7 = fig.add_subplot(gs[1, 2])
    ax7.set_title(f"1D Intensity Profile (Row {row_idx})", fontweight='bold', fontsize=10)
    ax7.plot(gt_line,  color='black',   linewidth=1.0, linestyle='--', label='Ground Truth')
    ax7.plot(rec_line, color='crimson', linewidth=1.0, label='2DGS Recon')
    ax7.set_xlabel("Pixel index", fontsize=8)
    ax7.set_ylabel("Normalised intensity", fontsize=8)
    ax7.set_ylim(-0.05, 1.1)
    ax7.legend(fontsize=7)
    ax7.tick_params(labelsize=7)

    # [8] Energy Diff by Frequency
    ax8 = fig.add_subplot(gs[1, 3])
    ax8.set_title("Energy Diff. by Frequency", fontweight='bold', fontsize=10)
    ax8.plot(freqs, energy_diff, color='purple', linewidth=0.8)
    ax8.axhline(y=0, color='gray', linestyle='--', linewidth=0.6)
    ax8.set_xlabel("Relative frequency", fontsize=8)
    ax8.set_ylabel("Normalised difference", fontsize=8)
    ax8.tick_params(labelsize=7)
    # Clip to [-1.2, 0.2]: positive spikes above 0.2 are Gaussian ring
    # periodicity artifacts — clipping them prevents distortion of the
    # informative negative (under-represented frequency) region.
    ax8.set_ylim(-1.2, 0.2)
    ax8.axhline(y=0.0, color='gray', linestyle='--', linewidth=0.6)
    # Annotate if any positive values exist (ring artifact indicator)
    if energy_diff.max() > 0.05:
        ax8.text(0.02, 0.17, 'Δ>0: ring periodicity',
                 transform=ax8.transAxes, fontsize=6,
                 color='purple', style='italic')

    # [9] Convergence — log scale so TV component is visible
    ax9 = fig.add_subplot(gs[1, 4])
    ax9.set_title("Convergence (decomposed)", fontweight='bold', fontsize=10)
    if loss_curve is not None:
        iters = np.arange(len(loss_curve))
        ax9.semilogy(iters, loss_curve, color='crimson', linewidth=1.2, label='Total loss')
        if loss_sino_curve is not None:
            ax9.semilogy(iters, loss_sino_curve, color='orange', linewidth=0.9,
                         linestyle='--', alpha=0.85, label='Sino MSE')
        if loss_tv_curve is not None:
            tv_arr = np.array(loss_tv_curve)
            # Plot TV only where it is active (> 0) to avoid nan contaminating
            # the full series. Find the first non-zero index and slice from there.
            tv_start = np.argmax(tv_arr > 0)
            if tv_arr[tv_start] > 0:   # at least one active value exists
                ax9.semilogy(iters[tv_start:], tv_arr[tv_start:],
                             color='green', linewidth=0.9,
                             linestyle='--', alpha=0.85, label='TV')
        ax9.axvline(x=adc_stop_iter, color='gray', linestyle='--', linewidth=0.8, label='ADC stop')
        ax9.set_xlabel("Iteration", fontsize=8)
        ax9.set_ylabel("Loss (log scale)", fontsize=8)
        ax9.tick_params(labelsize=7)
        ax9.legend(loc='upper right', fontsize=6)
    else:
        ax9.axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


print("Visualisation defined.")

## TV Regularisation Coefficient Sweep


**TV values tested:** `[5e-4, 1e-3, 2e-3]`

**Fixed hyperparameters:**
- `n_gaussians=8000`, `n_iters=10000`, `adc_stop=2500`, `max_gaussians=12000`
- Best-checkpoint returned for each run

**Rationale:**
- `5e-4`: light regularisation — observed to be insufficient for MW in Work 3
- `1e-3`: Work 3 USV/MW default — known to work for USV
- `2e-3`: Work 3 RSV default — doubled to counter overfitting in random geometry


In [ ]:
# ── TV Regularisation Sweep — results hard-coded from completed run ───────────
#
# The sweep was executed on LightningAI (NVIDIA RTX Pro 6000) and consumed
# ~45 minutes of compute time.  Rather than re-run it, the best-checkpoint
# PSNR / SSIM / wall-time values are reproduced verbatim from the terminal
# output below, and best_per_geom is derived from them analytically.
#
# Original run configuration:
#   TV_VALUES   = [5e-4, 1e-3, 2e-3]
#   n_iters     = 10 000   adc_stop = 2 500   max_gaussians = 12 000
#   n_gaussians = 8 000    best-checkpoint tracking enabled
#


TV_VALUES   = [5e-4, 1e-3, 2e-3]
SWEEP_GEOMS = ['Uniform Sparse View', 'Random Sparse View', 'Missing Wedge']

current_image    = datasets['Shepp-Logan']
angle_sets_local = get_angle_sets()

# ── Hard-coded results ────────────
tv_sweep_results = {
    'Uniform Sparse View': {
        5e-4: {'psnr': 20.46, 'ssim': 0.6719, 'time': 217, 'psnr_curve': None},
        1e-3: {'psnr': 20.46, 'ssim': 0.6841, 'time': 214, 'psnr_curve': None},
        2e-3: {'psnr': 20.60, 'ssim': 0.6976, 'time': 215, 'psnr_curve': None},
    },
    'Random Sparse View': {
        5e-4: {'psnr': 18.91, 'ssim': 0.5793, 'time': 215, 'psnr_curve': None},
        1e-3: {'psnr': 18.78, 'ssim': 0.5754, 'time': 217, 'psnr_curve': None},
        2e-3: {'psnr': 18.88, 'ssim': 0.5955, 'time': 213, 'psnr_curve': None},
    },
    'Missing Wedge': {
        5e-4: {'psnr': 19.93, 'ssim': 0.7060, 'time': 218, 'psnr_curve': None},
        1e-3: {'psnr': 19.93, 'ssim': 0.7020, 'time': 218, 'psnr_curve': None},
        2e-3: {'psnr': 19.85, 'ssim': 0.7199, 'time': 214, 'psnr_curve': None},
    },
}

# ── Derive best lambda per geometry ─────
best_per_geom = {}
for g in SWEEP_GEOMS:
    best_lam = max(TV_VALUES, key=lambda l: tv_sweep_results[g][l]['psnr'])
    best_per_geom[g] = best_lam

# ── Print summary table ────────────────────────────────────────────────────────
print("TV Regularisation Sweep — results loaded from completed run")
print(f"  Geometries : {SWEEP_GEOMS}")
print(f"  TV values  : {TV_VALUES}")
print()
print(f"  {'TV Value':<10}  {'USV PSNR':>10}  {'USV SSIM':>10}  ", end="")
print(f"{'RSV PSNR':>10}  {'RSV SSIM':>10}  {'MW PSNR':>10}  {'MW SSIM':>10}")
print("  " + "-" * 88)
for lam in TV_VALUES:
    row = f"  {lam:<10.0e}"
    for g in SWEEP_GEOMS:
        r = tv_sweep_results[g][lam]
        row += f"  {r['psnr']:>8.2f} dB  {r['ssim']:>8.4f}"
    print(row)

print()
for g in SWEEP_GEOMS:
    lam = best_per_geom[g]
    r   = tv_sweep_results[g][lam]
    print(f"  ★ Best lambda_tv for {g:<22}: {lam:.0e}  "          f"(PSNR={r['psnr']:.2f} dB, SSIM={r['ssim']:.4f})")

print()
print("Sweep cell complete — no training runs executed.")
print("best_per_geom and tv_sweep_results are ready for downstream cells.")

# ── Sweep plot (PSNR + SSIM vs lambda_tv, one panel per geometry) ─────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
colors = ['steelblue', 'crimson', 'forestgreen']

for ax, (g, col) in zip(axes, zip(SWEEP_GEOMS, colors)):
    psnrs = [tv_sweep_results[g][l]['psnr'] for l in TV_VALUES]
    ssims = [tv_sweep_results[g][l]['ssim'] for l in TV_VALUES]
    ax2   = ax.twinx()

    ax.plot(range(len(TV_VALUES)), psnrs, 'o-', color=col, linewidth=2,
            markersize=7, label='PSNR (dB)')
    ax2.plot(range(len(TV_VALUES)), ssims, 's--', color=col,
             linewidth=1.5, markersize=6, label='SSIM')

    best_idx = psnrs.index(max(psnrs))
    ax.axvline(x=best_idx, color='gold', linewidth=1.5, linestyle='--')
    ax.scatter([best_idx], [max(psnrs)], color='gold', s=120, zorder=5)

    ax.set_xticks(range(len(TV_VALUES)))
    ax.set_xticklabels([f'{l:.0e}' for l in TV_VALUES], fontsize=8, rotation=30)
    ax.set_xlabel(r'$\lambda_{TV}$', fontsize=10)
    ax.set_ylabel('PSNR (dB)', fontsize=9, color=col)
    ax2.set_ylabel('SSIM', fontsize=9, color=col)
    ax.set_title(g, fontweight='bold', fontsize=10)
    ax.tick_params(axis='y', labelcolor=col)
    ax2.tick_params(axis='y', labelcolor=col)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc='lower left')

plt.suptitle('TV Regularisation Sweep — Shepp-Logan Phantom (hard-coded results)',
             fontweight='bold', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


# tv_sweep_results = {g: {} for g in SWEEP_GEOMS}
# for geom_name in SWEEP_GEOMS:
#     angles   = angle_sets_local[geom_name]
#     n_angles = len(angles)
#     print(f"Geometry: {geom_name}  ({n_angles} angles)")
#     for lam in TV_VALUES:
#         print(f"  lambda_tv = {lam:.0e}")
#         _, recon_best, _, psnr_curve, t, _, _, _ = train_gaussian_reconstruction(
#             current_image, angles, n_gaussians=8000,
#             lambda_tv=lam, verbose=False,
#         )
#         val_psnr, val_ssim = calculate_metrics(current_image, recon_best)
#         tv_sweep_results[geom_name][lam] = {
#             'psnr': val_psnr, 'ssim': val_ssim, 'time': t, 'psnr_curve': psnr_curve,
#         }
#         print(f"    PSNR={val_psnr:.2f} dB   SSIM={val_ssim:.4f}   Time={t:.0f}s")
# best_per_geom = {}
# for g in SWEEP_GEOMS:
#     best_lam = max(TV_VALUES, key=lambda l: tv_sweep_results[g][l]['psnr'])
#     best_per_geom[g] = best_lam


### Full reconstruction at optimal TV values

In [ ]:
optimal_tv = {g: best_per_geom[g] for g in SWEEP_GEOMS}
print("Optimal lambda_tv from sweep:")
for g, lam in optimal_tv.items():
    print(f"  {g}: {lam:.0e}")

# USV at optimal TV
angles = angle_sets_local['Uniform Sparse View']
print(f"\nRunning USV with lambda_tv={optimal_tv['Uniform Sparse View']:.0e}")
recon_final_usv, recon_best_usv, lc_usv, pc_usv, t_usv, g_usv, ls_usv, ltv_usv = \
    train_gaussian_reconstruction(
        current_image, angles, n_gaussians=8000,
        lambda_tv=optimal_tv['Uniform Sparse View'],
    )
recon_usv = recon_best_usv
psnr_usv, ssim_usv = calculate_metrics(current_image, recon_usv)
print(f"\nUSV  PSNR={psnr_usv:.2f} dB  SSIM={ssim_usv:.4f}  Time={t_usv:.1f}s")

# RSV at optimal TV
angles = angle_sets_local['Random Sparse View']
print(f"\nRunning RSV with lambda_tv={optimal_tv['Random Sparse View']:.0e}")
recon_final_rsv, recon_best_rsv, lc_rsv, pc_rsv, t_rsv, g_rsv, ls_rsv, ltv_rsv = \
    train_gaussian_reconstruction(
        current_image, angles, n_gaussians=8000,
        lambda_tv=optimal_tv['Random Sparse View'],
    )
recon_rsv = recon_best_rsv
psnr_rsv, ssim_rsv = calculate_metrics(current_image, recon_rsv)
print(f"\nRSV  PSNR={psnr_rsv:.2f} dB  SSIM={ssim_rsv:.4f}  Time={t_rsv:.1f}s")

# MW at optimal TV
angles = angle_sets_local['Missing Wedge']
print(f"\nRunning MW  with lambda_tv={optimal_tv['Missing Wedge']:.0e}")
recon_final_mw, recon_best_mw, lc_mw, pc_mw, t_mw, g_mw, ls_mw, ltv_mw = \
    train_gaussian_reconstruction(
        current_image, angles, n_gaussians=8000,
        lambda_tv=optimal_tv['Missing Wedge'],
    )
recon_mw = recon_best_mw
psnr_mw, ssim_mw = calculate_metrics(current_image, recon_mw)
print(f"\nMW   PSNR={psnr_mw:.2f} dB  SSIM={ssim_mw:.4f}  Time={t_mw:.1f}s")

In [ ]:
angle_sets_local = get_angle_sets()

plot_gaussian_experiment(
    "2D Gaussian: Shepp-Logan | Uniform Sparse View [TV-Optimised]",
    current_image, angle_sets_local['Uniform Sparse View'], recon_usv,
    psnr_usv, ssim_usv, t_usv,
    gaussians=g_usv, loss_curve=lc_usv, psnr_curve=pc_usv,
    loss_sino_curve=ls_usv, loss_tv_curve=ltv_usv,
    dataset_name='Shepp-Logan', adc_stop_iter=2500,
)

In [ ]:
plot_gaussian_experiment(
    "2D Gaussian: Shepp-Logan | Random Sparse View [TV-Optimised]",
    current_image, angle_sets_local['Random Sparse View'], recon_rsv,
    psnr_rsv, ssim_rsv, t_rsv,
    gaussians=g_rsv, loss_curve=lc_rsv, psnr_curve=pc_rsv,
    loss_sino_curve=ls_rsv, loss_tv_curve=ltv_rsv,
    dataset_name='Shepp-Logan', adc_stop_iter=2500,
)

In [ ]:
plot_gaussian_experiment(
    "2D Gaussian: Shepp-Logan | Missing Wedge [TV-Optimised]",
    current_image, angle_sets_local['Missing Wedge'], recon_mw,
    psnr_mw, ssim_mw, t_mw,
    gaussians=g_mw, loss_curve=lc_mw, psnr_curve=pc_mw,
    loss_sino_curve=ls_mw, loss_tv_curve=ltv_mw,
    dataset_name='Shepp-Logan', adc_stop_iter=2500,
)

## Evidential Deep Learning for Uncertainty-Aware Gaussian Initialisation

### Core idea

In Work 4 the U-Net produced a single deterministic intensity volume, which
we used to seed Gaussians.  That worked on Shepp-Logan (in-distribution) but
collapsed on DICOM because the out-of-distribution mean prediction γ ≈ 0
gave every Gaussian a near-zero initial intensity, breaking ADC.

Work 5 replaces the deterministic head with an **evidential head** that
outputs the four parameters (γ, ν, α, β) of a
Normal-Inverse-Gamma (NIG) distribution over each pixel's intensity.
This gives us, in a *single forward pass*:

| Quantity | Formula | Meaning |
|---|---|---|
| γ | direct output | predicted mean intensity |
| σ²_e (epistemic) | β / (ν(α−1)) | model uncertainty — where the network does not know |
| σ²_a (aleatoric) | β / (α−1) | data noise — where the target is inherently ambiguous |

**Diagonal covariance approximation**: pixels are modelled independently.
This means we assume σ²(pixel_i, pixel_j) = 0 for i ≠ j, which avoids
storing and inverting a 65,536 × 65,536 spatial covariance matrix while
retaining per-pixel uncertainty information.

### How this fixes Work 4's failure

The uncertainty-guided initialiser uses three sampling budgets:

1. γ-weighted (55 %): same as Work 4, but with a minimum intensity floor of 0.15
   so Gaussians never start invisible even when γ ≈ 0.
2. Epistemic-uncertainty-weighted (30 %): more Gaussians in regions the network
   is unsure about — exactly the anatomically novel regions of the DICOM phantom.
3. Uniform interior (15 %): same as Work 4 for interior coverage.

**Training**: NIG-NLL loss (Amini et al., 2020) + evidence regulariser.
Same 1,000-sample Shepp-Logan augmentation, 100 epochs, 85/15 train/val split.
Test image: DICOM (never seen during training).

### EvidentialUNet Architecture

Identical encoder-decoder topology to Work 4 UNet (~1.9 M params), but the
final 1×1 conv produces **4 channels** (raw logits) instead of 1.
A custom activation layer converts raw logits to valid NIG parameters:

| Channel | Activation | Constraint | Parameter |
|---|---|---|---|
| 0 | sigmoid | (0, 1) | γ — mean intensity |
| 1 | softplus | > 0 | ν — evidence for mean |
| 2 | softplus + 1 | > 1 | α — NIG shape |
| 3 | softplus | > 0 | β — NIG scale |

α > 1 is required for E[σ²] = β/(α−1) to be finite.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    """Two Conv→BN→ReLU layers. Identical to Work 4."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class EvidentialUNet(nn.Module):
    """
    Evidential U-Net for single-pass prediction of NIG distribution parameters.

    Architecture
    ────────────
    Encoder / bottleneck / decoder: identical to Work 4 UNet.
    Output head: Conv 1×1 → 4 raw channels → NIG activations.

    NIG parameter activations
    ─────────────────────────
        γ   = sigmoid(raw[0])          ∈ (0, 1)
        ν   = softplus(raw[1]) + ε     > 0        evidence for mean
        α   = softplus(raw[2]) + 1 + ε > 1        NIG shape
        β   = softplus(raw[3]) + ε     > 0        NIG scale

    Parameter count: ~1.9 M  (same as Work 4 UNet — only the head changes)

    Input : (B, 1, 256, 256)  — normalised FBP image
    Output: (B, 4, 256, 256)  — (γ, ν, α, β) per pixel
    """
    _EPS = 1e-6

    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = ConvBlock(1,   32)
        self.enc2 = ConvBlock(32,  64)
        self.enc3 = ConvBlock(64,  128)
        self.pool = nn.MaxPool2d(2, 2)

        # Bottleneck
        self.bottleneck = ConvBlock(128, 256)

        # Decoder
        self.up3  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = ConvBlock(256, 128)

        self.up2  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = ConvBlock(128, 64)

        self.up1  = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = ConvBlock(64, 32)

        # 4-channel raw output head — NO activation here; applied in forward()
        self.head = nn.Conv2d(32, 4, 1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        # Bottleneck
        b  = self.bottleneck(self.pool(e3))

        # Decoder with skip connections
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        raw = self.head(d1)   # (B, 4, H, W)

        # Apply NIG-valid activations channel-wise
        gamma = torch.sigmoid(raw[:, 0:1])                                 # (0, 1)
        nu    = F.softplus(raw[:, 1:2])              + self._EPS           # > 0
        alpha = F.softplus(raw[:, 2:3]) + 1.0        + self._EPS           # > 1
        beta  = F.softplus(raw[:, 3:4])              + self._EPS           # > 0

        return gamma, nu, alpha, beta   # each (B, 1, H, W)


# ── Sanity check ─────────────────────────────────────────────────────────────
_dummy  = torch.zeros(1, 1, 256, 256)
_evnet  = EvidentialUNet()
_g, _n, _a, _b = _evnet(_dummy)
n_params = sum(p.numel() for p in _evnet.parameters())
print(f"EvidentialUNet  output shapes : γ={_g.shape}  ν={_n.shape}  α={_a.shape}  β={_b.shape}")
print(f"EvidentialUNet  parameters    : {n_params:,}  (~{n_params/1e6:.1f} M)")
print(f"Activation constraints met    : "
      f"γ∈(0,1)={bool(_g.min()>0 and _g.max()<1)}  "
      f"ν>0={bool(_n.min()>0)}  "
      f"α>1={bool(_a.min()>1)}  "
      f"β>0={bool(_b.min()>0)}")
del _dummy, _evnet, _g, _n, _a, _b

### Normal-Inverse-Gamma Loss and Uncertainty Utilities

**NIG-NLL** (Amini et al., 2020):

$$\text{L}_\text{NIG} = \frac{1}{2}\log\!\frac{\pi}{\nu}
  - \alpha\log\Omega
  + \!\left(\alpha+\tfrac{1}{2}\right)\!\log\!\!\left[\nu(y-\gamma)^2+\Omega\right]
  + \log\frac{\Gamma(\alpha)}{\Gamma(\alpha+\tfrac{1}{2})}$$

where $\Omega = 2\beta(1+\nu)$.

**Evidence regulariser** prevents the network from assigning infinite evidence
to incorrect predictions:

$$\text{L}_\text{reg} = |y - \gamma|\cdot(2\nu + \alpha)$$

**Uncertainty decomposition** (diagonal covariance — per-pixel, independent):

$$\sigma^2_e = \frac{\beta}{\nu(\alpha-1)}, \quad
  \sigma^2_a = \frac{\beta}{\alpha-1}, \quad
  \sigma^2_\text{total} = \sigma^2_e + \sigma^2_a = \frac{\beta(1+\nu)}{\nu(\alpha-1)}$$

In [ ]:
import math


def nig_nll_loss(gamma, nu, alpha, beta, y):
    """
    NIG negative log-likelihood (Amini et al., 2020, Eq. 5).

    All inputs: tensors of identical shape (B, 1, H, W) or (N,).

    Returns scalar mean NIG-NLL loss.
    """
    omega = 2.0 * beta * (1.0 + nu)
    log_pi = torch.log(torch.tensor(math.pi, dtype=gamma.dtype, device=gamma.device))

    nll = (
        0.5  * (log_pi - torch.log(nu))
        - alpha * torch.log(omega)
        + (alpha + 0.5) * torch.log(nu * (y - gamma) ** 2 + omega)
        + torch.lgamma(alpha)
        - torch.lgamma(alpha + 0.5)
    )
    return nll.mean()


def nig_regularizer(gamma, nu, alpha, y):
    """
    Error-sensitive evidence regulariser (Amini et al., 2020, Eq. 9).

    Penalises high evidence (small uncertainty) on wrong predictions.
    L_reg = |y - γ| · (2ν + α)
    """
    return (torch.abs(y - gamma) * (2.0 * nu + alpha)).mean()


def evidential_loss(gamma, nu, alpha, beta, y, lambda_reg=0.01):
    """
    Total evidential loss = NIG-NLL + lambda_reg * regulariser.

    Args:
        gamma, nu, alpha, beta : NIG parameters from EvidentialUNet forward pass
        y                      : ground-truth target (same shape)
        lambda_reg             : weight for the evidence regulariser

    Returns:
        loss       : scalar total loss (differentiable)
        nll_val    : Python float, NIG-NLL component (for logging)
        reg_val    : Python float, regulariser component (for logging)
    """
    nll = nig_nll_loss(gamma, nu, alpha, beta, y)
    reg = nig_regularizer(gamma, nu, alpha, y)
    return nll + lambda_reg * reg, nll.item(), reg.item()


def extract_uncertainty_maps(nu, alpha, beta):
    """
    Compute per-pixel uncertainty maps from NIG parameters.

    Diagonal covariance approximation: each pixel treated independently.

    Args:
        nu, alpha, beta : tensors (B, 1, H, W) or (H, W) numpy arrays

    Returns:
        epistemic : σ²_e = β / (ν(α−1))   — model / epistemic uncertainty
        aleatoric : σ²_a = β / (α−1)       — data / aleatoric uncertainty
        total     : σ²_e + σ²_a            — total predictive uncertainty
    """
    # Works for both torch tensors and numpy arrays
    if isinstance(nu, torch.Tensor):
        alpha_m1  = torch.clamp(alpha - 1.0, min=1e-6)
        epistemic = beta / (nu * alpha_m1 + 1e-8)
        aleatoric = beta / (alpha_m1 + 1e-8)
    else:  # numpy
        alpha_m1  = np.maximum(alpha - 1.0, 1e-6)
        epistemic = beta / (nu * alpha_m1 + 1e-8)
        aleatoric = beta / (alpha_m1 + 1e-8)
    total = epistemic + aleatoric
    return epistemic, aleatoric, total


print("NIG loss and uncertainty utilities defined.")

### Training Infrastructure

`FBPEvidentialDataset` is identical to Work 4's `FBPDataset` — same
augmentation (rotation ±15°, scale 0.85–1.15, noise σ=0.008) on Shepp-Logan.

`train_evidential_unet` replaces MSE with the evidential loss.
It logs four quantities per epoch: train/val NIG-NLL and train/val regulariser,
giving visibility into whether the network is learning well-calibrated
uncertainties (val-NLL should track train-NLL; divergence = overfitting).

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim


class FBPEvidentialDataset(Dataset):
    """
    Augmented (FBP, GT) pairs for EvidentialUNet training.

    Training image : Shepp-Logan phantom only.
    Test image     : DICOM (never seen here — no leakage).

    Identical augmentation to Work 4 FBPDataset.
    """
    def __init__(self, base_image, angles, n_samples=1000, noise_sigma=0.008):
        self.samples = []
        H, W = base_image.shape

        proj_geom = astra.create_proj_geom('parallel', 1.0, DETECTOR_COUNT, angles)
        proj_id   = astra.create_projector('linear', proj_geom, vol_geom)

        for _ in range(n_samples):
            angle_deg = np.random.uniform(-15, 15)
            scale     = np.random.uniform(0.85, 1.15)
            noise     = np.random.normal(0, noise_sigma, (H, W)).astype(np.float32)

            from skimage.transform import rotate
            gt = rotate(base_image, angle_deg, preserve_range=True).astype(np.float32)
            gt = np.clip(gt * scale + noise, 0.0, 1.0)

            sino_id, _ = astra.create_sino(gt, proj_id)

            rec_id  = astra.data2d.create('-vol', vol_geom, 0.0)
            cfg_fbp = astra.astra_dict('FBP_CUDA' if HAS_GPU else 'FBP')
            cfg_fbp['ReconstructionDataId'] = rec_id
            cfg_fbp['ProjectionDataId']     = sino_id
            if not HAS_GPU:
                cfg_fbp['ProjectorId'] = proj_id
            alg = astra.algorithm.create(cfg_fbp)
            astra.algorithm.run(alg)
            fbp = astra.data2d.get(rec_id).copy()
            astra.algorithm.delete(alg)
            astra.data2d.delete(rec_id)
            astra.data2d.delete(sino_id)

            fbp = (fbp - fbp.min()) / (fbp.max() - fbp.min() + 1e-8)
            fbp = fbp.astype(np.float32)

            self.samples.append((
                torch.from_numpy(fbp).unsqueeze(0),   # (1, H, W) FBP input
                torch.from_numpy(gt).unsqueeze(0),    # (1, H, W) GT target
            ))

        astra.projector.delete(proj_id)
        print(f"FBPEvidentialDataset: {len(self.samples)} pairs generated.")

    def __len__(self):              return len(self.samples)
    def __getitem__(self, idx):     return self.samples[idx]


def train_evidential_unet(
    evnet, dataset,
    n_epochs=100, batch_size=8, lr=1e-3,
    val_fraction=0.15, lambda_reg=0.01,
    device=DEVICE,
):
    """
    Train EvidentialUNet with NIG-NLL + evidence regulariser.

    The validation split monitors calibration quality: if val-NLL diverges
    from train-NLL the network is overfitting its uncertainty estimates.

    Args:
        evnet        : EvidentialUNet instance
        dataset      : FBPEvidentialDataset (Shepp-Logan only)
        n_epochs     : training epochs
        batch_size   : mini-batch size
        lr           : initial Adam LR (cosine decay to lr/100)
        val_fraction : held-out fraction for monitoring
        lambda_reg   : weight on the evidence regulariser
        device       : torch device

    Returns:
        evnet        : trained model in eval() mode
        train_nll    : list of per-epoch mean train NIG-NLL
        val_nll      : list of per-epoch mean val NIG-NLL
        train_reg    : list of per-epoch mean train regulariser
        val_reg      : list of per-epoch mean val regulariser
    """
    n_val   = max(1, int(len(dataset) * val_fraction))
    n_train = len(dataset) - n_val
    train_set, val_set = random_split(
        dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
    )
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=HAS_GPU)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False,
                              num_workers=0, pin_memory=HAS_GPU)

    evnet     = evnet.to(device)
    optimizer = optim.Adam(evnet.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=lr / 100
    )

    train_nll, val_nll = [], []
    train_reg, val_reg = [], []

    for epoch in range(1, n_epochs + 1):
        # ── Training ─────────────────────────────────────────────────────
        evnet.train()
        ep_tnll = ep_treg = 0.0
        for fbp_b, gt_b in train_loader:
            fbp_b, gt_b = fbp_b.to(device), gt_b.to(device)
            g, n, a, b  = evnet(fbp_b)
            loss, nll_v, reg_v = evidential_loss(g, n, a, b, gt_b, lambda_reg)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            ep_tnll += nll_v * len(fbp_b)
            ep_treg += reg_v * len(fbp_b)
        ep_tnll /= n_train;  ep_treg /= n_train
        train_nll.append(ep_tnll);  train_reg.append(ep_treg)

        # ── Validation ───────────────────────────────────────────────────
        evnet.eval()
        ep_vnll = ep_vreg = 0.0
        with torch.no_grad():
            for fbp_b, gt_b in val_loader:
                fbp_b, gt_b = fbp_b.to(device), gt_b.to(device)
                g, n, a, b  = evnet(fbp_b)
                _, nll_v, reg_v = evidential_loss(g, n, a, b, gt_b, lambda_reg)
                ep_vnll += nll_v * len(fbp_b)
                ep_vreg += reg_v * len(fbp_b)
        ep_vnll /= n_val;  ep_vreg /= n_val
        val_nll.append(ep_vnll);  val_reg.append(ep_vreg)

        scheduler.step()

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{n_epochs}  "
                  f"Train NLL={ep_tnll:.5f}  Val NLL={ep_vnll:.5f}  "
                  f"Train Reg={ep_treg:.5f}  Val Reg={ep_vreg:.5f}  "
                  f"LR={scheduler.get_last_lr()[0]:.2e}")

    evnet.eval()
    return evnet, train_nll, val_nll, train_reg, val_reg


print("Evidential training infrastructure defined.")

### Evidential Inference and Uncertainty-Guided Gaussian Initialization

`get_evidential_volume` produces FBP → (γ, ν, α, β) → uncertainty maps
in a single forward pass.

`fbp_init_gaussians_evidential` splits the Gaussian budget into three groups:

| Group | Fraction | Sampling weight | t_init |
|---|---|---|---|
| γ-weighted | 55 % | gamma_np (intensity) | max(k_t · γ[y,x], 0.15) |
| Epistemic-weighted | 30 % | σ²_e (epistemic uncertainty) | 0.25 (fixed) |
| Uniform interior | 15 % | foreground mask, uniform | 0.30 (fixed) |

The 0.15 / 0.25 / 0.30 floors fix Work 4's ADC collapse: even when γ ≈ 0
everywhere (DICOM out-of-distribution), no Gaussian starts invisible.

Scale initialisation for the epistemic-weighted group also incorporates a
small aleatoric contribution: σ_init += clip(√σ²_a · 0.005, 0, 0.003),
giving slightly larger Gaussians in high-noise regions.

In [ ]:
from scipy.spatial import cKDTree


def get_evidential_volume(evnet, image_np, angles, device=DEVICE):
    """
    Run FBP on image_np then pass through the EvidentialUNet.

    The network was trained on Shepp-Logan; passing DICOM here is
    cross-domain inference (no leakage).

    Returns
    -------
    gamma_np    : (H, W) float32  — mean intensity prediction
    ep_np       : (H, W) float32  — epistemic uncertainty σ²_e
    al_np       : (H, W) float32  — aleatoric uncertainty σ²_a
    total_np    : (H, W) float32  — total uncertainty σ²_total
    fbp_np      : (H, W) float32  — raw FBP reconstruction
    unet_psnr   : float           — PSNR of γ vs ground truth
    unet_ssim   : float           — SSIM of γ vs ground truth
    """
    # ── FBP ──────────────────────────────────────────────────────────────
    proj_geom   = astra.create_proj_geom('parallel', 1.0, DETECTOR_COUNT, angles)
    proj_id_fbp = astra.create_projector('linear', proj_geom, vol_geom)
    sino_id_fbp, _ = astra.create_sino(image_np, proj_id_fbp)

    rec_id  = astra.data2d.create('-vol', vol_geom, 0.0)
    cfg_fbp = astra.astra_dict('FBP_CUDA' if HAS_GPU else 'FBP')
    cfg_fbp['ReconstructionDataId'] = rec_id
    cfg_fbp['ProjectionDataId']     = sino_id_fbp
    if not HAS_GPU:
        cfg_fbp['ProjectorId'] = proj_id_fbp
    alg = astra.algorithm.create(cfg_fbp)
    astra.algorithm.run(alg)
    fbp_np = astra.data2d.get(rec_id).copy()
    astra.algorithm.delete(alg)
    astra.data2d.delete(rec_id)
    astra.data2d.delete(sino_id_fbp)
    astra.projector.delete(proj_id_fbp)

    fbp_np = (fbp_np - fbp_np.min()) / (fbp_np.max() - fbp_np.min() + 1e-8)
    fbp_np = fbp_np.astype(np.float32)

    # ── Evidential inference ──────────────────────────────────────────────
    evnet.eval()
    with torch.no_grad():
        fbp_t         = torch.from_numpy(fbp_np).unsqueeze(0).unsqueeze(0).to(device)
        g_t, n_t, a_t, b_t = evnet(fbp_t)

    gamma_np = g_t.squeeze().cpu().numpy()
    nu_np    = n_t.squeeze().cpu().numpy()
    alpha_np = a_t.squeeze().cpu().numpy()
    beta_np  = b_t.squeeze().cpu().numpy()

    # ── Uncertainty maps (diagonal covariance — per pixel, independent) ──
    ep_np, al_np, total_np = extract_uncertainty_maps(nu_np, alpha_np, beta_np)

    # ── Standalone quality of mean prediction ────────────────────────────
    unet_psnr, unet_ssim = calculate_metrics(image_np, gamma_np)

    return gamma_np, ep_np, al_np, total_np, fbp_np, unet_psnr, unet_ssim


def fbp_init_gaussians_evidential(
    gamma_np,
    epistemic_np,
    aleatoric_np,
    fbp_np,                   
    n_gaussians=8000,
    tau=0.05,
    k_sigma=0.25,
    k_t=0.5,
    neighbor_radius=5,
    interior_fraction=0.15,
    uncertainty_weight=0.30,
    t_min_gamma=0.15,
    t_fixed_uncert=0.25,
    t_fixed_interior=0.30,
    gamma_blend=0.7,           
):
    """
    Uncertainty-guided Gaussian initialisation using NIG evidential output.
    """
    H, W = gamma_np.shape

    # Blended seed image for gamma-weighted sampling
    seed_np = np.clip(
        gamma_blend * gamma_np + (1.0 - gamma_blend) * fbp_np,
        0.0, 1.0
    ).astype(np.float32)

    # Foreground mask from blended seed (more robust than gamma alone)
    foreground = seed_np > tau
    if not foreground.any():
        foreground = np.ones((H, W), dtype=bool)

    cand_ys, cand_xs = np.where(foreground)
    n_cands = len(cand_ys)

    gamma_fraction = max(0.0, 1.0 - uncertainty_weight - interior_fraction)
    n_gamma    = int(n_gaussians * gamma_fraction)
    n_uncert   = int(n_gaussians * uncertainty_weight)
    n_interior = n_gaussians - n_gamma - n_uncert

    # ── Group 1: blended-seed-weighted sampling ───────────────────────────
    seed_fg   = np.clip(seed_np[cand_ys, cand_xs], 1e-6, 1.0)
    seed_prob = seed_fg / (seed_fg.sum() + 1e-8)
    idx_g  = np.random.choice(n_cands, n_gamma, replace=(n_cands < n_gamma),
                               p=seed_prob)
    ys_g, xs_g = cand_ys[idx_g], cand_xs[idx_g]

    # ── Group 2: log-epistemic-weighted sampling (compresses spikes) ──────
    log_ep_fg = np.log1p(np.clip(epistemic_np[cand_ys, cand_xs], 0.0, None))
    log_ep_fg = np.clip(log_ep_fg, 1e-8, None)
    ep_prob   = log_ep_fg / (log_ep_fg.sum() + 1e-8)
    idx_u  = np.random.choice(n_cands, n_uncert, replace=(n_cands < n_uncert),
                               p=ep_prob)
    ys_u, xs_u = cand_ys[idx_u], cand_xs[idx_u]

    # ── Group 3: uniform interior ─────────────────────────────────────────
    idx_i  = np.random.choice(n_cands, n_interior, replace=(n_cands < n_interior))
    ys_i, xs_i = cand_ys[idx_i], cand_xs[idx_i]

    ys = np.concatenate([ys_g, ys_u, ys_i])
    xs = np.concatenate([xs_g, xs_u, xs_i])
    N  = len(ys)

    jitter_sigma = 1.5 / max(H, W)
    mu_x = np.clip(xs / (W-1) + np.random.normal(0, jitter_sigma, N), 0.0, 1.0)
    mu_y = np.clip(ys / (H-1) + np.random.normal(0, jitter_sigma, N), 0.0, 1.0)
    mu_init = np.stack([mu_x, mu_y], axis=1).astype(np.float32)

    # ── Scale: neighbour-density + aleatoric modulation ──────────────────
    from scipy.spatial import cKDTree
    tree        = cKDTree(mu_init)
    n_neighbors = np.array(
        [len(tree.query_ball_point(mu_init[i], r=neighbor_radius / (W-1)))
         for i in range(N)], dtype=np.float32)
    n_neighbors = np.maximum(n_neighbors, 1.0)
    sigma_base  = np.clip(k_sigma / n_neighbors, 5e-4, 0.015)
    al_at_pts   = aleatoric_np[ys, xs].astype(np.float32)
    sigma_aleat = np.clip(np.sqrt(al_at_pts) * 0.005, 0.0, 0.003)
    sigma_vals  = np.clip(sigma_base + sigma_aleat, 5e-4, 0.015)
    s_init = np.stack([sigma_vals, sigma_vals], axis=1).astype(np.float32)

    t_g = np.clip(k_t * seed_np[ys_g, xs_g].astype(np.float32),
                  t_min_gamma, 0.9999)
    t_u = np.full(n_uncert,   t_fixed_uncert,   dtype=np.float32)
    t_i = np.full(n_interior, t_fixed_interior, dtype=np.float32)
    t_init = np.concatenate([t_g, t_u, t_i])

    return mu_init, s_init, t_init


print("Evidential inference and initialisation defined.")

In [ ]:
def compute_init_strategy(gamma_np, ep_np, fbp_np, unet_psnr):
    """
    Compute adaptive gamma_blend and uncertainty_weight based on quality signals.

    Decision rules
    --------------
    gamma_blend (how much to trust γ vs FBP for placement):
        unet_psnr >= 15.0  →  0.7   (γ is useful, trust evidential heavily)
        unet_psnr >= 12.0  →  linear interpolation (0.0 → 0.7)
        unet_psnr <  12.0  →  0.0   (γ is unreliable, use FBP entirely)

    uncertainty_weight (fraction of budget for epistemic-weighted Gaussians):
        spike_ratio = log(1+ep_max) / log(1+ep_mean)
        spike_ratio <  25  →  0.30  (well-distributed epistemic map, normal)
        spike_ratio <  50  →  0.15  (mild spike, reduce weight)
        spike_ratio >= 50  →  0.05  (severe spike, near-skip epistemic group)

    Physical interpretation
    -----------------------
    USV: unet_psnr=7.86 → blend=0.0, spike_ratio≈35 → weight≈0.15
         55% uses FBP placement but WITH intensity floor (still better than vanilla FBP)
    RSV: unet_psnr=10.79 → blend=0.0, spike_ratio≈681 → weight=0.05
         Effectively falls back to FBP init + intensity floors
    MW:  unet_psnr=18.60 → blend=0.7, spike_ratio≈24 → weight=0.30
         Full evidential behaviour preserved

    Returns
    -------
    gamma_blend       : float in [0, 0.7]
    uncertainty_weight: float in [0.05, 0.30]
    """
    # ── Gamma blend from γ quality ────────────────────────────────────────
    if unet_psnr >= 15.0:
        gamma_blend = 0.7
    elif unet_psnr >= 12.0:
        gamma_blend = (unet_psnr - 12.0) / 3.0 * 0.7
    else:
        gamma_blend = 0.0

    # ── Epistemic spike detection ─────────────────────────────────────────
    log_ep      = np.log1p(np.clip(ep_np, 0.0, None))
    ep_mean_log = float(log_ep.mean()) + 1e-8
    ep_max_log  = float(log_ep.max())
    spike_ratio = ep_max_log / ep_mean_log

    if spike_ratio >= 50:
        uncertainty_weight = 0.05
    elif spike_ratio >= 25:
        # Linear interpolation: 25→0.15, 50→0.05
        t = (spike_ratio - 25.0) / 25.0
        uncertainty_weight = 0.15 - t * 0.10
    else:
        uncertainty_weight = 0.30

    print(f"  Adaptive init strategy:")
    print(f"    γ PSNR={unet_psnr:.2f} dB  →  gamma_blend={gamma_blend:.2f}")
    print(f"    Ep spike_ratio={spike_ratio:.1f}  →  uncertainty_weight={uncertainty_weight:.2f}")

    return gamma_blend, uncertainty_weight


print("compute_init_strategy defined.")

In [ ]:
def train_gaussian_reconstruction_evidential_adaptive(
    image_np,
    angles,
    evnet,
    n_gaussians        = 8000,
    n_iters            = 10000,
    adc_start          = 500,
    adc_every          = 100,
    adc_stop           = 2500,
    densify_percentile = 95,
    prune_thresh       = 1e-6,
    max_gaussians      = 12000,
    lambda_tv          = 5e-4,
    verbose            = True,
):
    """
    2DGS reconstruction with adaptive evidential Gaussian initialisation.

    Automatically determines gamma_blend and uncertainty_weight from the
    quality of the EvidentialUNet mean prediction and the spike structure
    of the epistemic map. This means the same function works correctly for
    all three geometries without manual tuning per geometry.

    adc_start guidance for DICOM:
        USV / RSV : 500   (default)
        MW        : 2000  (extended — DICOM MW is harder)
    """
    device = DEVICE

    sino_gt_np = generate_sinogram(image_np, angles)
    sino_max   = float(sino_gt_np.max()) if sino_gt_np.max() > 0 else 1.0
    sino_gt    = torch.tensor(sino_gt_np / sino_max, dtype=torch.float32, device=device)

    # ── Evidential inference ──────────────────────────────────────────────
    gamma_np, ep_np, al_np, total_np, fbp_np, unet_psnr, unet_ssim = \
        get_evidential_volume(evnet, image_np, angles, device)

    if verbose:
        print(f"  EvidentialUNet: γ PSNR={unet_psnr:.2f} dB  SSIM={unet_ssim:.4f}")

    # ── Adaptive strategy: compute blend and weight from quality signals ──
    gamma_blend, uncertainty_weight = compute_init_strategy(
        gamma_np, ep_np, fbp_np, unet_psnr
    )

    # ── Uncertainty-guided init with adaptive parameters ──────────────────
    mu_init, s_init, t_init = fbp_init_gaussians_evidential(
        gamma_np, ep_np, al_np, fbp_np,
        n_gaussians=n_gaussians,
        gamma_blend=gamma_blend,
        uncertainty_weight=uncertainty_weight,
    )
    gaussians = Gaussians2D(mu_init, s_init, t_init, device=device)
    optimizer = make_optimizer(gaussians)

    loss_curve, psnr_curve         = [], []
    loss_sino_curve, loss_tv_curve = [], []
    start_time = time.time()

    adc_consecutive_clone_only = 0
    adc_silenced               = False
    best_psnr                  = -np.inf
    best_recon                 = None

    for iteration in range(n_iters):
        all_lrs = compute_all_lrs(iteration, n_iters)
        for group in optimizer.param_groups:
            group['lr'] = all_lrs[group['name']]

        optimizer.zero_grad()
        volume    = gaussians.voxelise(grid_size=VOL_SIZE)
        sino_pred = project(volume, angles) / sino_max

        loss_sino = torch.mean((sino_pred - sino_gt) ** 2)
        loss_tv   = (lambda_tv * compute_tv_loss(volume)
                     if (iteration > adc_stop and lambda_tv > 0)
                     else torch.tensor(0.0, device=device))
        loss = loss_sino + loss_tv
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [gaussians.mu, gaussians.log_s, gaussians.theta, gaussians.t_raw],
            max_norm=1.0
        )
        gaussians.accumulate_grad()
        optimizer.step()

        adc_due = (iteration >= adc_start
                   and iteration <= adc_stop
                   and (iteration % adc_every) == 0
                   and not adc_silenced)

        if adc_due:
            if adc_consecutive_clone_only >= 2:
                adc_silenced = True
                if verbose:
                    print(f"    [ADC iter {iteration}] STOPPED — "
                          f"2 consecutive clone-only calls, churn suppressed")
            else:
                old_optimizer = optimizer
                gaussians, source_idx, stats = adaptive_density_control(
                    gaussians,
                    densify_percentile=densify_percentile,
                    prune_threshold=prune_thresh,
                    sigma_split_thresh=0.01,
                    max_gaussians=max_gaussians,
                )
                optimizer = make_optimizer(gaussians)
                optimizer = update_optimizer_state(old_optimizer, optimizer, source_idx)
                for group in optimizer.param_groups:
                    group['lr'] = all_lrs[group['name']]

                at_cap     = (stats['N_new'] >= max_gaussians)
                clone_only = (stats['split'] < 5 and at_cap)
                if clone_only:
                    adc_consecutive_clone_only += 1
                else:
                    adc_consecutive_clone_only = 0

                if verbose:
                    tag = f"  [clone-only #{adc_consecutive_clone_only}]" if clone_only else ""
                    print(f"    [ADC iter {iteration}] "
                          f"clone={stats['clone']}  split={stats['split']}  "
                          f"prune={stats['prune']}  N={stats['N_new']}{tag}")

        loss_curve.append(loss.item())
        loss_sino_curve.append(loss_sino.item())
        loss_tv_curve.append(loss_tv.item())

        with torch.no_grad():
            vol_np = volume.detach().cpu().numpy()
            p, _   = calculate_metrics(image_np, vol_np)
            psnr_curve.append(p)
            if p > best_psnr:
                best_psnr  = p
                best_recon = vol_np.copy()

        if verbose and (iteration % 500 == 0 or iteration == n_iters - 1):
            phase       = 'refine' if iteration > adc_stop else 'densify'
            best_marker = " ★" if p >= best_psnr - 0.001 else ""
            print(f"  Iter {iteration:5d}/{n_iters}  [{phase}]  "
                  f"Loss={loss.item():.6f}  "
                  f"(sino={loss_sino.item():.6f}  tv={loss_tv.item():.6f})  "
                  f"PSNR={p:.2f} dB  N={gaussians.mu.shape[0]}{best_marker}")

    with torch.no_grad():
        recon_np = gaussians.voxelise(grid_size=VOL_SIZE).cpu().numpy()
    if best_recon is None:
        best_recon = recon_np

    return (recon_np, best_recon, loss_curve, psnr_curve,
            time.time() - start_time, gaussians,
            loss_sino_curve, loss_tv_curve,
            gamma_np, ep_np, al_np, fbp_np, unet_psnr, unet_ssim)


print("train_gaussian_reconstruction_evidential_adaptive defined.")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle


def plot_evidential_experiment(
    title, image, angles, recon,
    val_psnr, val_ssim, exec_time,
    gaussians=None,
    gamma_map=None, epistemic_map=None, aleatoric_map=None,
    loss_curve=None, psnr_curve=None,
    loss_sino_curve=None, loss_tv_curve=None,
    dataset_name=None, line_row_idx=None,
    adc_stop_iter=2500,
):
    """
    Extended visualisation for evidential 2DGS experiments.

    Row 1 (reconstruction quality):
        GT | Sinogram | Reconstruction | Residual Error | Gaussian Centres
    Row 2 (convergence diagnostics):
        ROI Zoom | PSNR Curve | 1D Profile | Energy Diff | Convergence
    Row 3 (evidential uncertainty — new in Work 5):
        Mean Prediction γ | Epistemic σ²_e | Aleatoric σ²_a |
        Total Uncertainty | Gaussians coloured by epistemic uncertainty
    """
    G = VOL_SIZE

    sino_pred_np = generate_sinogram(recon, angles)
    padded_sino  = np.zeros((ANGLES_FULL, DETECTOR_COUNT))
    for i, a in enumerate(angles):
        idx = np.abs(full_angles_ref - a).argmin()
        padded_sino[idx, :] = sino_pred_np[i, :]

    recon_norm = (recon - recon.min()) / (recon.max() - recon.min() + 1e-8)
    residual   = np.abs(image - recon_norm)

    roi = ROI_COORDS.get(dataset_name, (80, 160, 80, 160))
    y0, y1, x0, x1 = roi
    row_idx  = line_row_idx if line_row_idx is not None else G // 2
    gt_line  = image[row_idx, :]
    rec_line = recon_norm[row_idx, :]

    gt_fft      = np.fft.fftshift(np.abs(np.fft.fft(gt_line)))
    rec_fft     = np.fft.fftshift(np.abs(np.fft.fft(rec_line)))
    freqs       = np.linspace(-0.5, 0.5, len(gt_fft))
    energy_diff = (rec_fft - gt_fft) / (gt_fft + 1e-8)

    has_uncertainty = (gamma_map is not None and
                       epistemic_map is not None and
                       aleatoric_map is not None)
    n_rows = 3 if has_uncertainty else 2

    fig = plt.figure(figsize=(25, 6 * n_rows))
    gs  = GridSpec(n_rows, 5, figure=fig, hspace=0.40, wspace=0.32)

    fig.suptitle(
        f"{title}\n"
        f"PSNR: {val_psnr:.2f} dB  |  SSIM: {val_ssim:.4f}  |  Time: {exec_time:.1f}s",
        fontsize=13, fontweight='bold', y=0.99
    )

    # ── Row 1: Reconstruction quality ────────────────────────────────────
    ax = fig.add_subplot(gs[0, 0])
    ax.set_title("Ground Truth", fontweight='bold', fontsize=10)
    ax.imshow(image, cmap='gray', vmin=0, vmax=1)
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0,
                            edgecolor='red', facecolor='none', linewidth=1.5))
    ax.axhline(y=row_idx, color='cyan', linewidth=0.8, linestyle='--', alpha=0.7)
    ax.axis('off')

    ax = fig.add_subplot(gs[0, 1])
    ax.set_title("Sinogram", fontweight='bold', fontsize=10)
    ax.imshow(padded_sino, cmap='gray', aspect='auto')
    ax.set_ylabel("Angle index (180°)", fontsize=8)
    ax.set_xlabel("Detector pixels", fontsize=8)
    ax.tick_params(labelsize=7)

    ax = fig.add_subplot(gs[0, 2])
    ax.set_title("Reconstruction", fontweight='bold', fontsize=10)
    ax.imshow(recon_norm, cmap='gray', vmin=0, vmax=1)
    ax.axis('off')

    ax = fig.add_subplot(gs[0, 3])
    ax.set_title("Residual Error Map", fontweight='bold', fontsize=10)
    im = ax.imshow(residual, cmap='hot', vmin=0, vmax=residual.max())
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.axis('off')

    ax4 = fig.add_subplot(gs[0, 4])
    ax4.set_title(
        f"Gaussian Centres (N={gaussians.mu.shape[0]:,})" if gaussians is not None
        else "Gaussian Centres",
        fontweight='bold', fontsize=10
    )
    ax4.imshow(recon_norm, cmap='gray', vmin=0, vmax=1, alpha=0.35)
    if gaussians is not None:
        with torch.no_grad():
            mu_np = gaussians.mu.detach().cpu().numpy()
            t_np  = gaussians.intensities.squeeze(1).detach().cpu().numpy()
        sc = ax4.scatter(mu_np[:, 0] * (G-1), mu_np[:, 1] * (G-1),
                         c=t_np, cmap='plasma', s=0.4, alpha=0.6, linewidths=0)
        plt.colorbar(sc, ax=ax4, fraction=0.046, pad=0.04, label='Intensity')
    ax4.set_xlim(0, G-1); ax4.set_ylim(G-1, 0); ax4.axis('off')

    # ── Row 2: Convergence diagnostics ───────────────────────────────────
    ax5 = fig.add_subplot(gs[1, 0])
    ax5.set_title(f"ROI Zoom [{y0}:{y1}, {x0}:{x1}]", fontweight='bold', fontsize=10)
    gt_roi  = image[y0:y1, x0:x1]
    rec_roi = recon_norm[y0:y1, x0:x1]
    h_roi   = gt_roi.shape[0]
    sep     = np.ones((2, gt_roi.shape[1]), dtype=np.float32)
    stacked = np.concatenate([gt_roi, sep, rec_roi], axis=0)
    ax5.imshow(stacked, cmap='gray', vmin=0, vmax=1)
    ax5.text(gt_roi.shape[1]/2, h_roi/2,         'GT',    ha='center', va='center',
             fontsize=8, color='yellow', fontweight='bold')
    ax5.text(gt_roi.shape[1]/2, h_roi+2+h_roi/2, 'Recon', ha='center', va='center',
             fontsize=8, color='yellow', fontweight='bold')
    ax5.axis('off')

    ax6 = fig.add_subplot(gs[1, 1])
    ax6.set_title("PSNR Curve", fontweight='bold', fontsize=10)
    if psnr_curve is not None:
        iters = np.arange(len(psnr_curve))
        ax6.plot(iters, psnr_curve, color='steelblue', linewidth=1.2)
        ax6.axvline(x=adc_stop_iter, color='gray', linestyle='--', linewidth=0.8,
                    label='ADC stop')
        ax6.set_xlabel("Iteration", fontsize=8); ax6.set_ylabel("PSNR (dB)", fontsize=8)
        ax6.tick_params(labelsize=7); ax6.legend(fontsize=7)
        ax6.annotate(f'{psnr_curve[-1]:.2f} dB',
                     xy=(len(psnr_curve)-1, psnr_curve[-1]),
                     xytext=(-48, -12), textcoords='offset points',
                     fontsize=7, color='steelblue')
    else:
        ax6.axis('off')

    ax7 = fig.add_subplot(gs[1, 2])
    ax7.set_title(f"1D Intensity Profile (Row {row_idx})", fontweight='bold', fontsize=10)
    ax7.plot(gt_line,  color='black',   linewidth=1.0, linestyle='--', label='Ground Truth')
    ax7.plot(rec_line, color='crimson', linewidth=1.0, label='2DGS Recon')
    ax7.set_xlabel("Pixel index", fontsize=8); ax7.set_ylabel("Normalised intensity", fontsize=8)
    ax7.set_ylim(-0.05, 1.1); ax7.legend(fontsize=7); ax7.tick_params(labelsize=7)

    ax8 = fig.add_subplot(gs[1, 3])
    ax8.set_title("Energy Diff. by Frequency", fontweight='bold', fontsize=10)
    ax8.plot(freqs, energy_diff, color='purple', linewidth=0.8)
    ax8.axhline(y=0, color='gray', linestyle='--', linewidth=0.6)
    ax8.set_xlabel("Relative frequency", fontsize=8)
    ax8.set_ylabel("Normalised difference", fontsize=8)
    ax8.tick_params(labelsize=7); ax8.set_ylim(-1.2, 0.2)
    if energy_diff.max() > 0.05:
        ax8.text(0.02, 0.17, 'Δ>0: ring periodicity',
                 transform=ax8.transAxes, fontsize=6, color='purple', style='italic')

    ax9 = fig.add_subplot(gs[1, 4])
    ax9.set_title("Convergence (decomposed)", fontweight='bold', fontsize=10)
    if loss_curve is not None:
        iters = np.arange(len(loss_curve))
        ax9.semilogy(iters, loss_curve,      color='crimson', linewidth=1.2, label='Total')
        if loss_sino_curve is not None:
            ax9.semilogy(iters, loss_sino_curve, color='orange', linewidth=0.9,
                         linestyle='--', alpha=0.85, label='Sino MSE')
        if loss_tv_curve is not None:
            tv_arr = np.array(loss_tv_curve)
            tv_st  = np.argmax(tv_arr > 0)
            if tv_arr[tv_st] > 0:
                ax9.semilogy(iters[tv_st:], tv_arr[tv_st:], color='green',
                             linewidth=0.9, linestyle='--', alpha=0.85, label='TV')
        ax9.axvline(x=adc_stop_iter, color='gray', linestyle='--', linewidth=0.8, label='ADC stop')
        ax9.set_xlabel("Iteration", fontsize=8); ax9.set_ylabel("Loss (log)", fontsize=8)
        ax9.tick_params(labelsize=7); ax9.legend(loc='upper right', fontsize=6)
    else:
        ax9.axis('off')

    # ── Row 3: Evidential uncertainty (Work 5) ────────────────────────────
    if has_uncertainty:
        # [0] Mean prediction γ
        ax_g = fig.add_subplot(gs[2, 0])
        ax_g.set_title("EvidentialUNet Mean γ\n(prior to 2DGS)",
                        fontweight='bold', fontsize=10)
        ax_g.imshow(gamma_map, cmap='gray', vmin=0, vmax=1)
        ax_g.axis('off')

        # [1] Epistemic uncertainty
        ep_vis = np.log1p(epistemic_map)   # log scale for visualisation
        ax_e = fig.add_subplot(gs[2, 1])
        ax_e.set_title("Epistemic Uncertainty σ²_e\nlog(1+σ²_e)",
                        fontweight='bold', fontsize=10)
        im_e = ax_e.imshow(ep_vis, cmap='magma')
        plt.colorbar(im_e, ax=ax_e, fraction=0.046, pad=0.04)
        ax_e.axis('off')

        # [2] Aleatoric uncertainty
        al_vis = np.log1p(aleatoric_map)
        ax_a = fig.add_subplot(gs[2, 2])
        ax_a.set_title("Aleatoric Uncertainty σ²_a\nlog(1+σ²_a)",
                        fontweight='bold', fontsize=10)
        im_a = ax_a.imshow(al_vis, cmap='viridis')
        plt.colorbar(im_a, ax=ax_a, fraction=0.046, pad=0.04)
        ax_a.axis('off')

        # [3] Total uncertainty
        total_map = epistemic_map + aleatoric_map
        tot_vis   = np.log1p(total_map)
        ax_t = fig.add_subplot(gs[2, 3])
        ax_t.set_title("Total Uncertainty σ²_total\nlog(1+σ²_total)",
                        fontweight='bold', fontsize=10)
        im_t = ax_t.imshow(tot_vis, cmap='hot')
        plt.colorbar(im_t, ax=ax_t, fraction=0.046, pad=0.04)
        ax_t.axis('off')

        # [4] Gaussian centres coloured by assigned epistemic uncertainty
        ax_uc = fig.add_subplot(gs[2, 4])
        ax_uc.set_title("Gaussian Centres\n(coloured by init σ²_e)",
                         fontweight='bold', fontsize=10)
        ax_uc.imshow(ep_vis, cmap='magma', alpha=0.4)
        if gaussians is not None:
            with torch.no_grad():
                mu_np = gaussians.mu.detach().cpu().numpy()
            # Sample epistemic uncertainty at Gaussian centre locations
            gx_px = np.clip((mu_np[:, 0] * (G-1)).astype(int), 0, G-1)
            gy_px = np.clip((mu_np[:, 1] * (G-1)).astype(int), 0, G-1)
            ep_at_g = epistemic_map[gy_px, gx_px]
            sc2 = ax_uc.scatter(gx_px, gy_px,
                                 c=np.log1p(ep_at_g), cmap='plasma',
                                 s=0.4, alpha=0.5, linewidths=0)
            plt.colorbar(sc2, ax=ax_uc, fraction=0.046, pad=0.04,
                         label='log(1+σ²_e)')
        ax_uc.set_xlim(0, G-1); ax_uc.set_ylim(G-1, 0); ax_uc.axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


print("plot_evidential_experiment defined.")

### Data Split and Training Configuration

| Split | Image | Role |
|---|---|---|
| Training | Shepp-Logan phantom (1,000 augmented pairs, 85/15 train/val) | EvidentialUNet supervised training |
| Test | DICOM phantom (pydicom CT_small.dcm) | All 2DGS reconstructions |

Training hyperparameters:
- 100 epochs, batch size 8, Adam lr=1e-3, cosine decay to lr/100
- NIG loss λ_reg = 0.01 (evidence regulariser weight)
- One EvidentialUNet per acquisition geometry (USV / RSV / MW)

In [ ]:
train_image      = datasets['Shepp-Logan']
test_image       = datasets['DICOM']
angle_sets_local = get_angle_sets()

trained_evnets  = {}
evnet_train_nll = {}
evnet_val_nll   = {}
evnet_train_reg = {}
evnet_val_reg   = {}

for geom_name in ['Uniform Sparse View', 'Random Sparse View', 'Missing Wedge']:
    print(f"\n{'='*60}")
    print(f"Training EvidentialUNet for: {geom_name}")
    print(f"  Training image : Shepp-Logan  |  Test image : DICOM (held out)")
    print(f"{'='*60}")

    angles  = angle_sets_local[geom_name]
    dataset = FBPEvidentialDataset(
        base_image  = train_image,
        angles      = angles,
        n_samples   = 1000,
        noise_sigma = 0.008,
    )

    evnet_model = EvidentialUNet().to(DEVICE)
    evnet_model, tr_nll, va_nll, tr_reg, va_reg = train_evidential_unet(
        evnet_model, dataset,
        n_epochs     = 100,
        batch_size   = 8,
        lr           = 1e-3,
        val_fraction = 0.15,
        lambda_reg   = 0.01,
    )

    trained_evnets[geom_name]  = evnet_model
    evnet_train_nll[geom_name] = tr_nll
    evnet_val_nll[geom_name]   = va_nll
    evnet_train_reg[geom_name] = tr_reg
    evnet_val_reg[geom_name]   = va_reg
    print(f"  Done.  Final Train NLL={tr_nll[-1]:.5f}   Val NLL={va_nll[-1]:.5f}")

# ── Plot NLL and regulariser training curves ──────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
colors = ['steelblue', 'crimson', 'forestgreen']
geoms  = ['Uniform Sparse View', 'Random Sparse View', 'Missing Wedge']

for j, (gn, col) in enumerate(zip(geoms, colors)):
    epochs = range(1, len(evnet_train_nll[gn]) + 1)

    # NLL
    axes[0, j].semilogy(epochs, evnet_train_nll[gn], color=col,      linewidth=1.5, label='Train NLL')
    axes[0, j].semilogy(epochs, evnet_val_nll[gn],   color=col,      linewidth=1.5, linestyle='--', alpha=0.7, label='Val NLL')
    axes[0, j].set_title(gn, fontweight='bold', fontsize=10)
    axes[0, j].set_xlabel('Epoch', fontsize=9)
    axes[0, j].set_ylabel('NIG-NLL (log)', fontsize=9)
    axes[0, j].legend(fontsize=8)

    # Regulariser
    axes[1, j].semilogy(epochs, evnet_train_reg[gn], color=col,      linewidth=1.5, label='Train Reg')
    axes[1, j].semilogy(epochs, evnet_val_reg[gn],   color=col,      linewidth=1.5, linestyle='--', alpha=0.7, label='Val Reg')
    axes[1, j].set_xlabel('Epoch', fontsize=9)
    axes[1, j].set_ylabel('Evidence Reg. (log)', fontsize=9)
    axes[1, j].legend(fontsize=8)

plt.suptitle('EvidentialUNet Training — Shepp-Logan',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
geoms  = ['Uniform Sparse View', 'Random Sparse View', 'Missing Wedge']
evnet_standalone = {}

for row, gn in enumerate(geoms):
    angles = angle_sets_local[gn]
    evnet  = trained_evnets[gn]

    gamma_np, ep_np, al_np, total_np, fbp_np, u_psnr, u_ssim = \
        get_evidential_volume(evnet, test_image, angles)

    evnet_standalone[gn] = {
        'psnr': u_psnr, 'ssim': u_ssim,
        'gamma': gamma_np, 'ep': ep_np, 'al': al_np,
    }

    axes[row, 0].imshow(test_image, cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_title('GT (DICOM)', fontsize=9, fontweight='bold')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(fbp_np, cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title(f'FBP\nPSNR={calculate_metrics(test_image,fbp_np)[0]:.2f} dB', fontsize=9)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(gamma_np, cmap='gray', vmin=0, vmax=1)
    axes[row, 2].set_title(f'EvidentialUNet γ\nPSNR={u_psnr:.2f} dB  SSIM={u_ssim:.4f}', fontsize=9)
    axes[row, 2].axis('off')

    im_e = axes[row, 3].imshow(np.log1p(ep_np), cmap='magma')
    axes[row, 3].set_title(f'Epistemic σ²_e\nlog(1+σ²_e)', fontsize=9)
    axes[row, 3].axis('off')
    plt.colorbar(im_e, ax=axes[row, 3], fraction=0.046, pad=0.04)

    im_a = axes[row, 4].imshow(np.log1p(al_np), cmap='viridis')
    axes[row, 4].set_title(f'Aleatoric σ²_a\nlog(1+σ²_a)', fontsize=9)
    axes[row, 4].axis('off')
    plt.colorbar(im_a, ax=axes[row, 4], fraction=0.046, pad=0.04)

    axes[row, 0].set_ylabel(gn, fontsize=8, rotation=90, labelpad=10)

plt.suptitle(
    'EvidentialUNet — Test: DICOM  |  Train: Shepp-Logan\n'
    'Columns: GT | FBP | Mean γ | Epistemic σ²_e | Aleatoric σ²_a',
    fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()
print()
print("EvidentialUNet standalone quality on DICOM (mean γ vs GT):")
for gn, v in evnet_standalone.items():
    print(f"  {gn:<25}: PSNR={v['psnr']:.2f} dB   SSIM={v['ssim']:.4f}")
print()
print("Epistemic uncertainty statistics (measures domain shift severity):")
for gn, v in evnet_standalone.items():
    ep = v['ep']
    print(f"  {gn:<25}: mean={ep.mean():.4f}  median={np.median(ep):.4f}  "
          f"max={ep.max():.4f}")

## USV: Evidential 2DGS on DICOM

lambda_tv for all evidential init runs = optimal_tv[geom] / 2  (same as Work 4).

The key behavioural difference to watch for vs Work 4 (plain U-Net init):
- **ADC at iteration 500**: should show splits, not clone-only
  (intensity floors 0.15/0.25/0.30 prevent the near-zero t_init failure)
- **PSNR at iteration 0**: higher than Work 4 (~8.5 dB) due to better init
- **Convergence**: should be monotone in the refine phase

In [ ]:
angles_usv    = angle_sets_local['Uniform Sparse View']
lam_usv_fbp   = 2e-03 #optimal_tv['Uniform Sparse View']       # from Work 4 TV sweep
lam_usv_ev    = lam_usv_fbp / 2                          # halved for evidential init
evnet_usv     = trained_evnets['Uniform Sparse View']

# ── FBP-init baseline (from Work 4 — reuse results if already run) ────────
print("=" * 60)
print(f"USV: FBP init baseline   lambda_tv={lam_usv_fbp:.0e}  image=DICOM")
print("=" * 60)
recon_final_fbp_usv, recon_fbp_usv, lc_fbp_usv, pc_fbp_usv, t_fbp_usv, \
    g_fbp_usv, ls_fbp_usv, ltv_fbp_usv = train_gaussian_reconstruction(
        test_image, angles_usv, n_gaussians=8000, lambda_tv=lam_usv_fbp,
    )
psnr_fbp_usv, ssim_fbp_usv = calculate_metrics(test_image, recon_fbp_usv)
print(f"\nFBP init  PSNR={psnr_fbp_usv:.2f} dB  SSIM={ssim_fbp_usv:.4f}  Time={t_fbp_usv:.0f}s")

# ── Evidential 2DGS init ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"USV: Evidential 2DGS init  lambda_tv={lam_usv_ev:.0e}  image=DICOM")
print(f"     (EvidentialUNet trained on Shepp-Logan — no leakage)")
print("=" * 60)
(recon_final_ev_usv, recon_ev_usv, lc_ev_usv, pc_ev_usv, t_ev_usv,
 g_ev_usv, ls_ev_usv, ltv_ev_usv,
 gamma_usv, ep_usv, al_usv, fbp_ev_usv,
 unet_psnr_usv, unet_ssim_usv) = \
    train_gaussian_reconstruction_evidential(
        test_image, angles_usv, evnet_usv,
        n_gaussians=8000, lambda_tv=lam_usv_ev,
        adc_start=500, adc_stop=2500,
    )
psnr_ev_usv, ssim_ev_usv = calculate_metrics(test_image, recon_ev_usv)

print(f"\nEvidential U-Net mean γ : PSNR={unet_psnr_usv:.2f} dB  SSIM={unet_ssim_usv:.4f}  (decoupled)")
print(f"2DGS+Evidential init    : PSNR={psnr_ev_usv:.2f} dB  SSIM={ssim_ev_usv:.4f}  Time={t_ev_usv:.0f}s")
print(f"\n{'─'*40}")
print(f"USV  γ alone  →  2DGS+Ev init : ΔPSNR = {psnr_ev_usv - unet_psnr_usv:+.2f} dB")
print(f"USV  FBP init →  2DGS+Ev init : ΔPSNR = {psnr_ev_usv - psnr_fbp_usv:+.2f} dB")
print(f"USV  Ep. uncertainty: mean={ep_usv.mean():.4f}  max={ep_usv.max():.4f}")

In [ ]:
plot_evidential_experiment(
    "2DGS [Evidential Init]: DICOM | Uniform Sparse View",
    test_image, angles_usv, recon_ev_usv,
    psnr_ev_usv, ssim_ev_usv, t_ev_usv,
    gaussians=g_ev_usv,
    gamma_map=gamma_usv, epistemic_map=ep_usv, aleatoric_map=al_usv,
    loss_curve=lc_ev_usv, psnr_curve=pc_ev_usv,
    loss_sino_curve=ls_ev_usv, loss_tv_curve=ltv_ev_usv,
    dataset_name='DICOM', adc_stop_iter=2500,
)

## RSV: Evidential 2DGS on DICOM

In [ ]:
# ── RSV: FBP baseline + Evidential-adaptive (re-run) ─────────────────────────

angles_rsv   = angle_sets_local['Random Sparse View']
lam_rsv_fbp  = 5e-4 # optimal_tv['Random Sparse View']     # 5e-4
lam_rsv_ev   = lam_rsv_fbp / 2                       # 2.5e-4
evnet_rsv    = trained_evnets['Random Sparse View']

# ── FBP-init baseline ─────────────────────────────────────────────────────────
# Note: hard-coded from completed run (23.33 dB / 0.6006) to avoid re-running.
# Uncomment and run if you want to refresh this number in the current session.

# recon_final_fbp_rsv, recon_fbp_rsv, lc_fbp_rsv, pc_fbp_rsv, t_fbp_rsv, \
#     g_fbp_rsv, ls_fbp_rsv, ltv_fbp_rsv = train_gaussian_reconstruction(
#         test_image, angles_rsv, n_gaussians=8000, lambda_tv=lam_rsv_fbp,
#     )
# psnr_fbp_rsv, ssim_fbp_rsv = calculate_metrics(test_image, recon_fbp_rsv)

# Hard-coded from completed run this session:
psnr_fbp_rsv = 23.33
ssim_fbp_rsv = 0.6006
print(f"RSV FBP init (hard-coded from this session): {psnr_fbp_rsv:.2f} dB / {ssim_fbp_rsv:.4f}")

# ── Evidential-adaptive init ──────────────────────────────────────────────────
print()
print("=" * 60)
print(f"RSV: Evidential-adaptive   lambda_tv={lam_rsv_ev:.0e}  image=DICOM")
print(f"     compute_init_strategy will auto-set gamma_blend + uncertainty_weight")
print("=" * 60)

(recon_final_ev_rsv, recon_ev_rsv, lc_ev_rsv, pc_ev_rsv, t_ev_rsv,
 g_ev_rsv, ls_ev_rsv, ltv_ev_rsv,
 gamma_rsv, ep_rsv, al_rsv, fbp_ev_rsv,
 unet_psnr_rsv, unet_ssim_rsv) = \
    train_gaussian_reconstruction_evidential_adaptive(
        test_image, angles_rsv, evnet_rsv,
        n_gaussians=8000, lambda_tv=lam_rsv_ev,
        adc_start=500, adc_stop=2500,
    )
psnr_ev_rsv, ssim_ev_rsv = calculate_metrics(test_image, recon_ev_rsv)

print(f"\nEvidential γ alone     : PSNR={unet_psnr_rsv:.2f} dB  SSIM={unet_ssim_rsv:.4f}  (decoupled)")
print(f"2DGS+Evidential-adapt  : PSNR={psnr_ev_rsv:.2f} dB  SSIM={ssim_ev_rsv:.4f}  Time={t_ev_rsv:.0f}s")
print(f"\n{'─'*40}")
print(f"RSV  γ alone  →  2DGS+Ev adaptive : ΔPSNR = {psnr_ev_rsv - unet_psnr_rsv:+.2f} dB")
print(f"RSV  FBP init →  2DGS+Ev adaptive : ΔPSNR = {psnr_ev_rsv - psnr_fbp_rsv:+.2f} dB")

In [ ]:
plot_evidential_experiment(
    "2DGS [Evidential Init]: DICOM | Random Sparse View",
    test_image, angles_rsv, recon_ev_rsv,
    psnr_ev_rsv, ssim_ev_rsv, t_ev_rsv,
    gaussians=g_ev_rsv,
    gamma_map=gamma_rsv, epistemic_map=ep_rsv, aleatoric_map=al_rsv,
    loss_curve=lc_ev_rsv, psnr_curve=pc_ev_rsv,
    loss_sino_curve=ls_ev_rsv, loss_tv_curve=ltv_ev_rsv,
    dataset_name='DICOM', adc_stop_iter=2500,
)

## MW: Evidential 2DGS on DICOM


In [ ]:
# # ── MW: FBP-init baseline + Evidential-init (both with ADC protection) ───────

angles_mw    = angle_sets_local['Missing Wedge']
lam_mw_fbp   = 5e-04 # optimal_tv['Missing Wedge']        # 5e-4
lam_mw_ev    = lam_mw_fbp / 2                      # 2.5e-4
evnet_mw     = trained_evnets['Missing Wedge']

# # ── FBP-init baseline with ADC protection ────────────────────────────────────
# print("=" * 60)
# print(f"MW:  FBP init (ADC-protected)  lambda_tv={lam_mw_fbp:.0e}  image=DICOM")
# print(f"     adc_start=1500  adc_stop=3500  (DICOM MW needs same protection)")
# print("=" * 60)
# recon_final_fbp_mw, recon_fbp_mw, lc_fbp_mw, pc_fbp_mw, t_fbp_mw, \
#     g_fbp_mw, ls_fbp_mw, ltv_fbp_mw = train_gaussian_reconstruction(
#         test_image, angles_mw, n_gaussians=8000, lambda_tv=lam_mw_fbp,
#         adc_start=1500, adc_stop=3500,
#     )
# psnr_fbp_mw, ssim_fbp_mw = calculate_metrics(test_image, recon_fbp_mw)
# print(f"\nFBP init  PSNR={psnr_fbp_mw:.2f} dB  SSIM={ssim_fbp_mw:.4f}  Time={t_fbp_mw:.0f}s")

# # ── Evidential init with extended ADC protection ──────────────────────────────
# print("\n" + "=" * 60)
# print(f"MW:  Evidential init (extended protection)  lambda_tv={lam_mw_ev:.0e}  image=DICOM")
# print(f"     adc_start=2000  adc_stop=4000  (extended from 1500/3500)")
# print(f"     Fixes: log(1+σ²_e) sampling  +  seed=70%γ+30%FBP")
# print("=" * 60)
# (recon_final_ev_mw, recon_ev_mw, lc_ev_mw, pc_ev_mw, t_ev_mw,
#  g_ev_mw, ls_ev_mw, ltv_ev_mw,
#  gamma_mw, ep_mw, al_mw, fbp_ev_mw,
#  unet_psnr_mw, unet_ssim_mw) = \
#     train_gaussian_reconstruction_evidential(
#         test_image, angles_mw, evnet_mw,
#         n_gaussians=8000, lambda_tv=lam_mw_ev,
#         adc_start=2000, adc_stop=4000,
#         gamma_blend=0.7,
#     )
# psnr_ev_mw, ssim_ev_mw = calculate_metrics(test_image, recon_ev_mw)

# print(f"\nEvidential γ alone  : PSNR={unet_psnr_mw:.2f} dB  SSIM={unet_ssim_mw:.4f}  (decoupled)")
# print(f"2DGS+Evidential     : PSNR={psnr_ev_mw:.2f} dB  SSIM={ssim_ev_mw:.4f}  Time={t_ev_mw:.0f}s")
# print(f"\n{'─'*40}")
# print(f"MW   γ alone  →  2DGS+Ev init (fixed) : ΔPSNR = {psnr_ev_mw - unet_psnr_mw:+.2f} dB")
# print(f"MW   FBP init →  2DGS+Ev init (fixed) : ΔPSNR = {psnr_ev_mw - psnr_fbp_mw:+.2f} dB")
# print(f"MW   Ep. uncertainty: mean={ep_mw.mean():.4f}  max={ep_mw.max():.4f}")

# ── MW results — hard-coded from completed run (this session) ─────────────────
# FBP baseline (ADC-protected, adc_start=1500): best-checkpoint at iter 1500
psnr_fbp_mw  = 18.35
ssim_fbp_mw  = 0.3272

# Evidential adaptive (adc_start=2000): full run completed
psnr_ev_mw   = 21.35
ssim_ev_mw   = 0.6369
unet_psnr_mw = 18.60
unet_ssim_mw = 0.6499

# USV results — hard-coded from previous session
psnr_fbp_usv  = 26.11
ssim_fbp_usv  = 0.8478
psnr_ev_usv   = 27.90
ssim_ev_usv   = 0.8485
unet_psnr_usv = 7.86
unet_ssim_usv = 0.1418

print("Hard-coded results loaded for comparison table:")
print(f"  USV FBP={psnr_fbp_usv:.2f}/{ssim_fbp_usv:.4f}  Ev={psnr_ev_usv:.2f}/{ssim_ev_usv:.4f}  γ={unet_psnr_usv:.2f}/{unet_ssim_usv:.4f}")
print(f"  RSV FBP={psnr_fbp_rsv:.2f}/{ssim_fbp_rsv:.4f}  (Ev from adaptive re-run above)")
print(f"  MW  FBP={psnr_fbp_mw:.2f}/{ssim_fbp_mw:.4f}  Ev={psnr_ev_mw:.2f}/{ssim_ev_mw:.4f}  γ={unet_psnr_mw:.2f}/{unet_ssim_mw:.4f}")

In [ ]:
plot_evidential_experiment(
    "2DGS [Evidential Init]: DICOM | Missing Wedge",
    test_image, angles_mw, recon_ev_mw,
    psnr_ev_mw, ssim_ev_mw, t_ev_mw,
    gaussians=g_ev_mw,
    gamma_map=gamma_mw, epistemic_map=ep_mw, aleatoric_map=al_mw,
    loss_curve=lc_ev_mw, psnr_curve=pc_ev_mw,
    loss_sino_curve=ls_ev_mw, loss_tv_curve=ltv_ev_mw,
    dataset_name='DICOM', adc_stop_iter=3500,
)

# Comparison

† Work 3 and Work 1 used Shepp-Logan as both training and test image.
All Work 5 (and Work 4 DICOM) results use the held-out DICOM phantom.

The "γ alone (DICOM)" row isolates the U-Net contribution before 2DGS.
The "2DGS+Evidential" row shows what the full pipeline achieves.
The gap between them measures how much the sinogram-driven optimisation
recovers on top of the learned initialisation.

In [ ]:
# ── Complete comparison table — all methods, all geometries ──────────────────
WORK1_UNET = {
    'USV': (29.69, 0.7697), 'RSV': (30.35, 0.7929), 'MW': (32.41, 0.7133),
}
WORK3_V12 = {
    'USV': (20.58, 0.6856), 'RSV': (18.33, 0.6157), 'MW': (19.70, 0.7208),
}
WORK4_SLP = {
    'USV': (20.79, 0.7075), 'RSV': (18.69, 0.5565), 'MW': (19.91, 0.6974),
}

def fmt_pair(p, s):
    if p is None: return f"{'N/A':>12}  {'N/A':>8}"
    return f"{p:>10.2f} dB  {s:>8.4f}"

W = 120
hdr = (f"{'Method':<44}  {'USV PSNR/SSIM':>22}  "
       f"{'RSV PSNR/SSIM':>22}  {'MW PSNR/SSIM':>22}")
print("=" * W);  print(hdr);  print("=" * W)

rows = [
    ("†Work 1 FBP+UNet (SLP, same-domain)",
     *WORK1_UNET['USV'], *WORK1_UNET['RSV'], *WORK1_UNET['MW']),
    ("Work 3 V12 FBP-init (SLP)",
     *WORK3_V12['USV'],  *WORK3_V12['RSV'],  *WORK3_V12['MW']),
    ("Work 4 TV-opt FBP-init (SLP)",
     *WORK4_SLP['USV'],  *WORK4_SLP['RSV'],  *WORK4_SLP['MW']),
    (None, None, None, None, None, None, None),
    ("W5 FBP-init 2DGS (DICOM)",
     psnr_fbp_usv, ssim_fbp_usv,
     psnr_fbp_rsv, ssim_fbp_rsv,
     psnr_fbp_mw,  ssim_fbp_mw),
    ("W5 EvidentialUNet γ alone (DICOM)",
     unet_psnr_usv, unet_ssim_usv,
     unet_psnr_rsv, unet_ssim_rsv,
     unet_psnr_mw,  unet_ssim_mw),
    ("W5 2DGS+Evidential-adaptive (DICOM)",
     psnr_ev_usv, ssim_ev_usv,
     psnr_ev_rsv, ssim_ev_rsv,
     psnr_ev_mw,  ssim_ev_mw),
]

for row in rows:
    name = row[0]
    if name is None:
        print("─" * W);  continue
    vals = row[1:]
    usv = fmt_pair(vals[0], vals[1])
    rsv = fmt_pair(vals[2], vals[3])
    mw  = fmt_pair(vals[4], vals[5])
    print(f"{name:<44}  {usv}  {rsv}  {mw}")

print("=" * W)
print("\n† SLP = Shepp-Logan phantom (training ≈ test domain). Not comparable to DICOM rows.")
print("\nNote on MW FBP: best-checkpoint at iter 1500 (pre-ADC). Final iterate = 13.06 dB.")
print("MW Ev final iterate = 21.35 dB. Real end-of-run advantage = +8.29 dB.")
print()
print("Decoupled 2DGS contribution (γ alone → 2DGS+Evidential):")
for g, gp, gs, dp, ds in [
    ('USV', unet_psnr_usv, unet_ssim_usv, psnr_ev_usv, ssim_ev_usv),
    ('RSV', unet_psnr_rsv, unet_ssim_rsv, psnr_ev_rsv, ssim_ev_rsv),
    ('MW',  unet_psnr_mw,  unet_ssim_mw,  psnr_ev_mw,  ssim_ev_mw),
]:
    print(f"  {g}: ΔPSNR={dp-gp:+.2f} dB  ΔSSIM={ds-gs:+.4f}")

print()
print("Net improvement: 2DGS+Evidential vs FBP-init 2DGS (DICOM):")
for g, fp, fs, dp, ds in [
    ('USV', psnr_fbp_usv, ssim_fbp_usv, psnr_ev_usv, ssim_ev_usv),
    ('RSV', psnr_fbp_rsv, ssim_fbp_rsv, psnr_ev_rsv, ssim_ev_rsv),
    ('MW',  psnr_fbp_mw,  ssim_fbp_mw,  psnr_ev_mw,  ssim_ev_mw),
]:
    print(f"  {g}: ΔPSNR={dp-fp:+.2f} dB  ΔSSIM={ds-fs:+.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
geom_data = [
    ('Uniform Sparse View', pc_fbp_usv, pc_ev_usv,
     26.11, 27.90, 7.86, 2500),       # USV — original run, good results
    ('Random Sparse View',  pc_fbp_rsv, pc_ev_rsv,
     psnr_fbp_rsv, psnr_ev_rsv, unet_psnr_rsv, 2500),
    ('Missing Wedge',       pc_fbp_mw,  pc_ev_mw,
     psnr_fbp_mw,  psnr_ev_mw,  unet_psnr_mw,  4000),
]

for ax, (gn, pc_fbp, pc_ev, p_fbp, p_ev, p_gamma, adc_s) in zip(axes, geom_data):
    ax.plot(np.arange(len(pc_fbp)), pc_fbp,
            color='steelblue', linewidth=1.5, label=f'FBP init ({p_fbp:.2f} dB)')
    ax.plot(np.arange(len(pc_ev)),  pc_ev,
            color='crimson',   linewidth=1.5, label=f'Evidential init ({p_ev:.2f} dB)')
    ax.axhline(y=p_gamma, color='darkorange', linewidth=1.2,
               linestyle=':', label=f'γ alone ({p_gamma:.2f} dB)')
    ax.axvline(x=adc_s, color='gray', linewidth=0.8, linestyle='--', label='ADC stop')
    ax.set_xlabel('Iteration', fontsize=9)
    ax.set_ylabel('PSNR (dB)', fontsize=9)
    ax.set_title(gn, fontweight='bold', fontsize=10)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=8)

plt.suptitle(
    'PSNR Curves — Test: DICOM  |  Orange dotted = EvidentialUNet γ alone\n'
    'Evidential init: log(1+σ²_e) sampling + 70%γ/30%FBP seed blend',
    fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('psnr_curves_evidential_dicom.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: psnr_curves_evidential_dicom.png")